In [ ]:
import torch
import subprocess

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        print("VRAM:",
              round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2),
              "GB")
        print("Compute capability:",
              torch.cuda.get_device_capability(i))

    print("\nBF16 supported:", torch.cuda.is_bf16_supported())

subprocess.run(["nvidia-smi"])

In [ ]:
from pathlib import Path
import zipfile
import hashlib
import json

INPUT = Path("/kaggle/input/sceneops-training-artifacts")
archives = list(INPUT.glob("*.zip"))

assert len(archives) == 1, f"Expected one ZIP, found {len(archives)}"

zip_path = archives[0]
print("ZIP SHA256:", hashlib.sha256(zip_path.read_bytes()).hexdigest())

extract = Path("/kaggle/working/SceneOps_Training_Artifacts")
extract.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract)

checkpoint = extract / "checkpoint-223"
dataset = extract / "dataset"

state = json.loads(
    (checkpoint / "trainer_state.json").read_text()
)

print("\nGlobal step:", state["global_step"])
print("Epoch:", state["epoch"])
print("Max steps:", state["max_steps"])

print("\nCheckpoint files:", len(list(checkpoint.iterdir())))
print("Dataset files:", [p.name for p in dataset.glob("*.jsonl")])

assert state["global_step"] == 223
assert state["max_steps"] == 669
assert len(list(dataset.glob("*.jsonl"))) == 3
assert (checkpoint / "optimizer.pt").exists()
assert (checkpoint / "scheduler.pt").exists()
assert (checkpoint / "rng_state.pth").exists()

print("\nTRANSFER VERIFIED")

In [ ]:
from pathlib import Path
import zipfile
import hashlib
import json

INPUT = Path("/kaggle/input/sceneops-training-artifacts")
archives = list(INPUT.glob("*.zip"))

assert len(archives) == 1, f"Expected one ZIP, found {len(archives)}"

zip_path = archives[0]
print("ZIP SHA256:", hashlib.sha256(zip_path.read_bytes()).hexdigest())

extract = Path("/kaggle/working/SceneOps_Training_Artifacts")
extract.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract)

checkpoint = extract / "checkpoint-223"
dataset = extract / "dataset"

state = json.loads(
    (checkpoint / "trainer_state.json").read_text()
)

print("\nGlobal step:", state["global_step"])
print("Epoch:", state["epoch"])
print("Max steps:", state["max_steps"])

print("\nCheckpoint files:", len(list(checkpoint.iterdir())))
print("Dataset files:", [p.name for p in dataset.glob("*.jsonl")])

assert state["global_step"] == 223
assert state["max_steps"] == 669
assert len(list(dataset.glob("*.jsonl"))) == 3
assert (checkpoint / "optimizer.pt").exists()
assert (checkpoint / "scheduler.pt").exists()
assert (checkpoint / "rng_state.pth").exists()

print("\nTRANSFER VERIFIED")

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("=== KAGGLE INPUT ROOT ===")
print("Exists:", root.exists())

for item in root.iterdir():
    print("\n", item.name, "| Directory:", item.is_dir())

    if item.is_dir():
        for child in list(item.iterdir())[:20]:
            print("   ", child.name, "| Directory:", child.is_dir())

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*"):
    print(p)

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/notebooks/priyanshu4404/sceneops-training-v2")

print("=== DATASET CONTENTS ===")

for p in ROOT.rglob("*"):
    print(p.relative_to(ROOT), "| Directory:", p.is_dir())

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/notebooks/priyanshu4404/sceneops-training-v2")

print("=== DATASET CONTENTS ===")

for p in ROOT.rglob("*"):
    print(p.relative_to(ROOT), "| Directory:", p.is_dir())

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("=== ALL ZIP FILES ===")
zips = list(root.rglob("*.zip"))

for p in zips:
    print(p)

print("\n=== ALL JSONL FILES ===")
jsonls = list(root.rglob("*.jsonl"))

for p in jsonls:
    print(p)

print("\n=== ALL INPUT DIRECTORIES ===")
for p in root.iterdir():
    print(p)

In [ ]:
from pathlib import Path
import json

ROOT = Path(
    "/kaggle/input/datasets/priyanshu4404/"
    "sceneops-training-artifacts/SceneOps_Training_Artifacts"
)

CHECKPOINT = ROOT / "checkpoint-223"
DATASET = ROOT / "dataset"

print("=== CHECKPOINT FILES ===")
for p in sorted(CHECKPOINT.iterdir()):
    print(p.name, round(p.stat().st_size / 1024**2, 2), "MB")

print("\n=== TRAINING STATE ===")
state = json.loads(
    (CHECKPOINT / "trainer_state.json").read_text()
)

print("Global step:", state["global_step"])
print("Epoch:", state["epoch"])
print("Max steps:", state["max_steps"])

print("\n=== DATASET COUNTS ===")
for p in sorted(DATASET.glob("*.jsonl")):
    with p.open(encoding="utf-8") as f:
        count = sum(1 for _ in f)
    print(p.name, ":", count)

required = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "optimizer.pt",
    "scheduler.pt",
    "rng_state.pth",
    "trainer_state.json",
    "training_args.bin",
]

missing = [
    f for f in required
    if not (CHECKPOINT / f).exists()
]

print("\nMissing checkpoint files:", missing)

assert not missing
assert state["global_step"] == 223
assert state["max_steps"] == 669

print("\nCHECKPOINT READY FOR RESUME")

In [ ]:
from pathlib import Path
import shutil
import json
import torch
import importlib.metadata as metadata

# Read-only Kaggle input
ROOT = Path(
    "/kaggle/input/datasets/priyanshu4404/"
    "sceneops-training-artifacts/SceneOps_Training_Artifacts"
)

SOURCE_CHECKPOINT = ROOT / "checkpoint-223"
DATASET_DIR = ROOT / "dataset"

# Writable working directory
WORK = Path("/kaggle/working/SceneOps")
CHECKPOINT_DIR = WORK / "checkpoints"
CHECKPOINT = CHECKPOINT_DIR / "checkpoint-223"

WORK.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Copy checkpoint only if not already copied
if not CHECKPOINT.exists():
    shutil.copytree(SOURCE_CHECKPOINT, CHECKPOINT)

# Verify copied checkpoint
state = json.loads(
    (CHECKPOINT / "trainer_state.json").read_text()
)

assert state["global_step"] == 223
assert state["max_steps"] == 669

print("=== WRITABLE CHECKPOINT ===")
print(CHECKPOINT)
print("Global step:", state["global_step"])
print("Epoch:", state["epoch"])
print("Max steps:", state["max_steps"])

print("\n=== DATASET PATH ===")
print(DATASET_DIR)

# Package versions
print("\n=== ENVIRONMENT VERSIONS ===")
for package in [
    "torch",
    "transformers",
    "trl",
    "peft",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "tokenizers",
]:
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "NOT INSTALLED")

print("\n=== GPU ===")
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", torch.cuda.is_bf16_supported())

# Inspect original saved training arguments
args_path = CHECKPOINT / "training_args.bin"

if args_path.exists():
    original_args = torch.load(
        args_path,
        map_location="cpu",
        weights_only=False
    )

    print("\n=== SAVED TRAINING CONFIG ===")
    for key in [
        "num_train_epochs",
        "per_device_train_batch_size",
        "gradient_accumulation_steps",
        "learning_rate",
        "warmup_steps",
        "optim",
        "fp16",
        "bf16",
        "gradient_checkpointing",
        "seed",
    ]:
        print(f"{key}: {getattr(original_args, key, None)}")

print("\nPREPARATION COMPLETE")

In [ ]:
%pip install --no-cache-dir \
    "transformers==4.57.1" \
    "trl==0.24.0" \
    "peft==0.18.0" \
    "accelerate==1.12.0" \
    "bitsandbytes==0.49.2"

In [ ]:
import torch
import transformers
import trl
import peft
import accelerate
import bitsandbytes as bnb

print("=== PACKAGE VERIFICATION ===")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bnb.__version__)

print("\n=== GPU VERIFICATION ===")
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

assert torch.cuda.is_available()
assert torch.cuda.is_bf16_supported()

print("\nALL IMPORTS AND GPU CHECKS PASSED")

In [ ]:
from pathlib import Path
import shutil
import json

SOURCE = Path(
    "/kaggle/input/datasets/priyanshu4404/sceneops-training-artifacts/"
    "SceneOps_Training_Artifacts"
)

WORK = Path("/kaggle/working/SceneOps")
CHECKPOINT_SOURCE = SOURCE / "checkpoint-223"
CHECKPOINT = WORK / "checkpoints" / "checkpoint-223"
DATASET = SOURCE / "dataset"

assert CHECKPOINT_SOURCE.exists(), f"Checkpoint not found: {CHECKPOINT_SOURCE}"
assert DATASET.exists(), f"Dataset not found: {DATASET}"

WORK.mkdir(parents=True, exist_ok=True)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT.exists():
    shutil.copytree(CHECKPOINT_SOURCE, CHECKPOINT)

with open(CHECKPOINT / "trainer_state.json", "r") as f:
    state = json.load(f)

print("=== CHECKPOINT VERIFICATION ===")
print("Checkpoint:", CHECKPOINT)
print("Global step:", state["global_step"])
print("Epoch:", state["epoch"])
print("Max steps:", state["max_steps"])
print("Checkpoint files:", len(list(CHECKPOINT.iterdir())))

print("\n=== DATASET VERIFICATION ===")
for name in ["train_v4.jsonl", "validation_v4.jsonl", "test_v4.jsonl"]:
    path = DATASET / name
    assert path.exists(), f"Missing: {name}"
    with open(path, "r", encoding="utf-8") as f:
        count = sum(1 for _ in f)
    print(f"{name}: {count} examples")

assert state["global_step"] == 223
assert state["max_steps"] == 669

print("\nCHECKPOINT AND DATASET READY")

In [ ]:
from pathlib import Path

print("=== DATASET DIRECTORY ===")
print("Path:", DATASET)
print("Exists:", DATASET.exists())

for path in sorted(DATASET.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(DATASET)}  |  {path.stat().st_size:,} bytes")

In [ ]:
import json

DATASET_FILES = {
    "train": DATASET / "sceneops_narration_train_v4.jsonl",
    "validation": DATASET / "sceneops_narration_validation_v4.jsonl",
    "test": DATASET / "sceneops_narration_test_v4.jsonl",
}

expected_counts = {
    "train": 3554,
    "validation": 1121,
    "test": 1181,
}

print("=== DATASET VERIFICATION ===")

for split, path in DATASET_FILES.items():
    assert path.exists(), f"Missing file: {path}"

    with open(path, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    print(f"{split}: {len(rows)} examples | Expected: {expected_counts[split]}")
    assert len(rows) == expected_counts[split], f"{split} count mismatch"

    if rows:
        print("  First example keys:", list(rows[0].keys()))

print("\nALL DATASET SPLITS VERIFIED")

In [ ]:
import torch
from pathlib import Path

args_path = CHECKPOINT / "training_args.bin"

print("Loading saved training arguments...")

saved_args = torch.load(
    args_path,
    map_location="cpu",
    weights_only=False
)

print("\n=== SAVED TRAINING CONFIGURATION ===")

fields = [
    "output_dir",
    "num_train_epochs",
    "per_device_train_batch_size",
    "per_device_eval_batch_size",
    "gradient_accumulation_steps",
    "learning_rate",
    "lr_scheduler_type",
    "warmup_steps",
    "weight_decay",
    "optim",
    "max_length",
    "bf16",
    "fp16",
    "gradient_checkpointing",
    "save_strategy",
    "eval_strategy",
    "report_to",
    "seed",
]

for field in fields:
    print(f"{field}: {getattr(saved_args, field, 'NOT FOUND')}")

print("\nSaved argument class:", type(saved_args))
print("TRAINING ARGUMENTS LOADED")

In [ ]:
import json
from pathlib import Path

state_path = CHECKPOINT / "trainer_state.json"

with open(state_path, "r") as f:
    trainer_state = json.load(f)

print("=== RESUME STATE ===")
for key in [
    "global_step",
    "epoch",
    "max_steps",
    "best_metric",
    "best_model_checkpoint",
]:
    print(f"{key}: {trainer_state.get(key)}")

print("\n=== REQUIRED RESUME FILES ===")
required_files = [
    "adapter_model.safetensors",
    "adapter_config.json",
    "optimizer.pt",
    "scheduler.pt",
    "rng_state.pth",
    "trainer_state.json",
    "training_args.bin",
]

for name in required_files:
    path = CHECKPOINT / name
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

assert all((CHECKPOINT / f).exists() for f in required_files)
assert trainer_state["global_step"] == 223

print("\nRESUME STATE VERIFIED")

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = load_jsonl(DATASET_FILES["train"])
val_rows = load_jsonl(DATASET_FILES["validation"])
test_rows = load_jsonl(DATASET_FILES["test"])

# Validate conversation structure
for split_name, rows in [
    ("train", train_rows),
    ("validation", val_rows),
    ("test", test_rows),
]:
    for i, row in enumerate(rows):
        assert "messages" in row, f"{split_name}[{i}] missing messages"
        messages = row["messages"]
        assert isinstance(messages, list) and len(messages) >= 2
        assert messages[-1]["role"] == "assistant"
        assert all("role" in m and "content" in m for m in messages)

train_dataset = Dataset.from_list(train_rows)
val_dataset = Dataset.from_list(val_rows)
test_dataset = Dataset.from_list(test_rows)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\n=== SAMPLE MESSAGE STRUCTURE ===")
for message in train_rows[0]["messages"]:
    print(f"\n[{message['role']}]")
    print(message["content"][:250])

print("\nDATASETS LOADED AND VALIDATED")

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("\nGPU memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
print("TOKENIZER, MODEL AND LoRA READY")

In [ ]:
from dataclasses import dataclass
import torch

@dataclass
class SceneOpsDataCollator:
    tokenizer: object
    max_length: int = 512

    def __call__(self, features):
        input_ids_list = []
        labels_list = []

        for feature in features:
            messages = feature["messages"]

            prompt = messages[:-1]
            answer = messages[-1]

            prompt_ids = self.tokenizer.apply_chat_template(
                prompt,
                tokenize=True,
                add_generation_prompt=True,
            )

            full_ids = self.tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=False,
            )

            if len(full_ids) > self.max_length:
                full_ids = full_ids[:self.max_length]

            labels = full_ids.copy()
            prompt_length = min(len(prompt_ids), len(labels))
            labels[:prompt_length] = [-100] * prompt_length

            input_ids_list.append(full_ids)
            labels_list.append(labels)

        batch_max = max(len(ids) for ids in input_ids_list)

        padded_ids = []
        padded_labels = []
        attention_masks = []

        for ids, labels in zip(input_ids_list, labels_list):
            pad_len = batch_max - len(ids)

            padded_ids.append(
                ids + [self.tokenizer.pad_token_id] * pad_len
            )
            padded_labels.append(
                labels + [-100] * pad_len
            )
            attention_masks.append(
                [1] * len(ids) + [0] * pad_len
            )

        return {
            "input_ids": torch.tensor(padded_ids, dtype=torch.long),
            "labels": torch.tensor(padded_labels, dtype=torch.long),
            "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        }

data_collator = SceneOpsDataCollator(tokenizer=tokenizer, max_length=512)

print("Custom collator initialized.")

In [ ]:
import torch

# Test one batch
test_batch = data_collator(train_rows[:2])

input_ids = test_batch["input_ids"]
labels = test_batch["labels"]
attention_mask = test_batch["attention_mask"]

print("=== COLLATOR TEST ===")
print("Input shape:", tuple(input_ids.shape))
print("Labels shape:", tuple(labels.shape))
print("Attention mask shape:", tuple(attention_mask.shape))

for i in range(len(input_ids)):
    supervised = (labels[i] != -100).sum().item()
    masked = (labels[i] == -100).sum().item()
    print(f"\nSample {i + 1}:")
    print("  Supervised tokens:", supervised)
    print("  Masked tokens:", masked)
    print("  Total tokens:", attention_mask[i].sum().item())

assert input_ids.shape == labels.shape == attention_mask.shape
assert torch.all(labels[attention_mask == 0] == -100)
assert all((labels[i] != -100).any() for i in range(len(labels)))

print("\nBATCH SHAPES AND LABEL MASKING PASSED")

In [ ]:
sample = train_rows[0]["messages"]

prompt_ids = tokenizer.apply_chat_template(
    sample[:-1],
    tokenize=True,
    add_generation_prompt=True,
)

full_ids = tokenizer.apply_chat_template(
    sample,
    tokenize=True,
    add_generation_prompt=False,
)

print("Prompt tokens:", len(prompt_ids))
print("Full conversation tokens:", len(full_ids))
print("Assistant target tokens:", len(full_ids) - len(prompt_ids))
print("Assistant begins at:", len(prompt_ids))
print("Assistant target preview:")
print(tokenizer.decode(full_ids[len(prompt_ids):len(prompt_ids)+80]))

assert full_ids[:len(prompt_ids)] == prompt_ids
print("\nPROMPT BOUNDARY VERIFIED")

In [ ]:
from dataclasses import dataclass
import torch

@dataclass
class SceneOpsDataCollator:
    tokenizer: object
    max_length: int = 512

    def __call__(self, features):
        batch_ids = []
        batch_labels = []

        for feature in features:
            messages = feature["messages"]

            prompt_ids = self.tokenizer.apply_chat_template(
                messages[:-1],
                tokenize=True,
                add_generation_prompt=True,
            )

            full_ids = self.tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=False,
            )

            assert full_ids[:len(prompt_ids)] == prompt_ids

            # Preserve the assistant target; trim prompt only if necessary.
            target_ids = full_ids[len(prompt_ids):]
            if len(target_ids) >= self.max_length:
                raise ValueError("Assistant target exceeds max_length.")

            available_prompt = self.max_length - len(target_ids)
            prompt_ids = prompt_ids[-available_prompt:] if available_prompt else []

            input_ids = prompt_ids + target_ids
            labels = [-100] * len(prompt_ids) + target_ids

            batch_ids.append(input_ids)
            batch_labels.append(labels)

        max_len = max(len(x) for x in batch_ids)

        input_ids = [
            x + [self.tokenizer.pad_token_id] * (max_len - len(x))
            for x in batch_ids
        ]
        labels = [
            x + [-100] * (max_len - len(x))
            for x in batch_labels
        ]
        attention_mask = [
            [1] * len(x) + [0] * (max_len - len(x))
            for x in batch_ids
        ]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }

data_collator = SceneOpsDataCollator(tokenizer, max_length=512)

test_batch = data_collator(train_rows[:2])
assert test_batch["input_ids"].shape == test_batch["labels"].shape
assert torch.all(test_batch["labels"][test_batch["attention_mask"] == 0] == -100)

print("Final collator test passed.")
print("Batch shape:", tuple(test_batch["input_ids"].shape))
print("Assistant supervised tokens:", (test_batch["labels"] != -100).sum(dim=1).tolist())

In [ ]:
from trl import SFTConfig

OUTPUT_DIR = "/kaggle/working/SceneOps/checkpoints"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    optim="adamw_torch",

    max_length=512,
    packing=False,
    dataset_kwargs={"skip_prepare_dataset": True},

    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    remove_unused_columns=False,
    dataloader_num_workers=0,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=10,
    report_to="none",
    run_name="SceneOps-Qwen2.5-1.5B-QLoRA",
    seed=42,
)

print("=== TRAINING CONFIGURATION ===")
print("Output directory:", training_args.output_dir)
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Learning rate:", training_args.learning_rate)
print("Max sequence length:", training_args.max_length)
print("BF16 trainer autocast:", training_args.bf16)
print("Evaluation:", training_args.eval_strategy)
print("Checkpoint saving:", training_args.save_strategy)
print("W&B:", training_args.report_to)

assert training_args.output_dir == OUTPUT_DIR
assert training_args.num_train_epochs == 3

print("\nTRAINING CONFIGURATION READY")

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("=== TRAINER INITIALIZED ===")
print("Current global step:", trainer.state.global_step)
print("Expected resume step:", 223)
print("Total planned steps:", trainer.args.max_steps)
print("Train examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))

assert trainer.state.global_step == 0
assert len(trainer.train_dataset) == 3554
assert len(trainer.eval_dataset) == 1121

print("\nTRAINER READY FOR CHECKPOINT RESUME")

In [ ]:
print("=== PRE-RESUME CHECK ===")
print("Calculated max steps:", trainer.state.max_steps)
print("Training dataloader batches:", len(trainer.get_train_dataloader()))
print("Gradient accumulation:", trainer.args.gradient_accumulation_steps)
print("Checkpoint global step:", trainer_state["global_step"])
print("Checkpoint max steps:", trainer_state["max_steps"])
print("Current trainer global step:", trainer.state.global_step)

assert trainer.state.max_steps == 669, (
    f"Step mismatch: expected 669, got {trainer.state.max_steps}"
)
assert trainer_state["max_steps"] == trainer.state.max_steps

print("\nSTEP COUNTS MATCH — READY TO RESUME")

In [ ]:
print("=== ORIGINAL VS CURRENT TRAINING SETTINGS ===")

for key in [
    "max_steps",
    "num_train_epochs",
    "per_device_train_batch_size",
    "gradient_accumulation_steps",
    "world_size",
    "dataloader_drop_last",
    "group_by_length",
    "length_column_name",
]:
    print(
        f"{key}: "
        f"saved={getattr(saved_args, key, 'N/A')} | "
        f"current={getattr(training_args, key, 'N/A')}"
    )

print("\nSaved checkpoint step:", trainer_state["global_step"])
print("Saved checkpoint max steps:", trainer_state["max_steps"])
print("Current dataloader batches:", len(trainer.get_train_dataloader()))

In [ ]:
print("=== DATALOADER DIAGNOSTICS ===")

loader = trainer.get_train_dataloader()

print("Dataset examples:", len(train_dataset))
print("Dataloader batches:", len(loader))
print("Dataloader batch_size:", loader.batch_size)
print("Trainer train_batch_size:", trainer.args.train_batch_size)
print("Per-device batch size:", trainer.args.per_device_train_batch_size)
print("Internal train batch size:", getattr(trainer, "_train_batch_size", "N/A"))
print("Gradient accumulation:", trainer.args.gradient_accumulation_steps)
print("World size:", trainer.args.world_size)
print("Drop last:", loader.drop_last)

print("\n=== CHECKPOINT REFERENCE ===")
print("Saved global step:", trainer_state["global_step"])
print("Saved max steps:", trainer_state["max_steps"])
print("Saved epoch:", trainer_state["epoch"])

In [ ]:
print("=== GPU / ACCELERATOR DIAGNOSTICS ===")
print("CUDA devices:", torch.cuda.device_count())
print("Trainer n_gpu:", trainer.args.n_gpu)
print("Trainer device:", trainer.args.device)
print("Parallel mode:", trainer.args.parallel_mode)
print("Accelerator processes:", trainer.accelerator.num_processes)
print("Accelerator split_batches:", trainer.accelerator.split_batches)
print("Internal batch size:", trainer._train_batch_size)
print("Effective train batch:", trainer.args.train_batch_size)

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

print("Visible GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", torch.cuda.is_bf16_supported())

assert torch.cuda.device_count() == 1
assert torch.cuda.is_bf16_supported()

print("\nSINGLE GPU CONFIGURATION VERIFIED")

In [ ]:
import os
import json
import shutil
import torch

from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
)
from trl import SFTConfig, SFTTrainer

SOURCE = Path(
    "/kaggle/input/datasets/priyanshu4404/sceneops-training-artifacts/"
    "SceneOps_Training_Artifacts"
)

WORK = Path("/kaggle/working/SceneOps")
DATASET = SOURCE / "dataset"
CHECKPOINT_SOURCE = SOURCE / "checkpoint-223"
CHECKPOINT = WORK / "checkpoints" / "checkpoint-223"

WORK.mkdir(parents=True, exist_ok=True)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT.exists():
    shutil.copytree(CHECKPOINT_SOURCE, CHECKPOINT)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = load_jsonl(DATASET / "sceneops_narration_train_v4.jsonl")
val_rows = load_jsonl(DATASET / "sceneops_narration_validation_v4.jsonl")
test_rows = load_jsonl(DATASET / "sceneops_narration_test_v4.jsonl")

train_dataset = Dataset.from_list(train_rows)
val_dataset = Dataset.from_list(val_rows)
test_dataset = Dataset.from_list(test_rows)

with open(CHECKPOINT / "trainer_state.json") as f:
    trainer_state = json.load(f)

print("GPU:", torch.cuda.get_device_name(0))
print("Visible GPUs:", torch.cuda.device_count())
print("Checkpoint step:", trainer_state["global_step"])
print("Checkpoint max steps:", trainer_state["max_steps"])
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Checkpoint files:", len(list(CHECKPOINT.iterdir())))

assert torch.cuda.device_count() == 1
assert trainer_state["global_step"] == 223
assert trainer_state["max_steps"] == 669
assert len(train_dataset) == 3554
assert len(val_dataset) == 1121

print("\nRESTORE SUCCESSFUL")

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map={"": 0},
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("\nGPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
print("MODEL READY")

In [ ]:
from dataclasses import dataclass
import torch
from trl import SFTConfig

@dataclass
class SceneOpsDataCollator:
    tokenizer: object
    max_length: int = 512

    def __call__(self, features):
        batch_ids, batch_labels = [], []

        for feature in features:
            messages = feature["messages"]

            prompt_ids = self.tokenizer.apply_chat_template(
                messages[:-1],
                tokenize=True,
                add_generation_prompt=True,
            )

            full_ids = self.tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=False,
            )

            assert full_ids[:len(prompt_ids)] == prompt_ids

            target_ids = full_ids[len(prompt_ids):]

            if len(target_ids) >= self.max_length:
                raise ValueError("Assistant target exceeds max_length")

            available_prompt = self.max_length - len(target_ids)
            prompt_ids = prompt_ids[-available_prompt:] if available_prompt else []

            batch_ids.append(prompt_ids + target_ids)
            batch_labels.append([-100] * len(prompt_ids) + target_ids)

        max_len = max(map(len, batch_ids))

        input_ids = [
            x + [self.tokenizer.pad_token_id] * (max_len - len(x))
            for x in batch_ids
        ]
        labels = [
            x + [-100] * (max_len - len(x))
            for x in batch_labels
        ]
        attention_mask = [
            [1] * len(x) + [0] * (max_len - len(x))
            for x in batch_ids
        ]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }

data_collator = SceneOpsDataCollator(tokenizer)

training_args = SFTConfig(
    output_dir="/kaggle/working/SceneOps/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    optim="adamw_torch",
    max_length=512,
    packing=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    dataloader_num_workers=0,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    report_to="none",
    run_name="SceneOps-Qwen2.5-1.5B-QLoRA",
    seed=42,
)

print("Collator and training configuration restored.")

In [ ]:
import math
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

batches_per_epoch = len(trainer.get_train_dataloader())
steps_per_epoch = math.ceil(
    batches_per_epoch / training_args.gradient_accumulation_steps
)
expected_total_steps = steps_per_epoch * int(training_args.num_train_epochs)

print("=== SINGLE GPU TRAINER VERIFICATION ===")
print("Visible GPUs:", torch.cuda.device_count())
print("Trainer n_gpu:", trainer.args.n_gpu)
print("Internal batch size:", trainer._train_batch_size)
print("Dataloader batches per epoch:", batches_per_epoch)
print("Optimizer steps per epoch:", steps_per_epoch)
print("Expected total steps:", expected_total_steps)
print("Checkpoint resume step:", trainer_state["global_step"])
print("Checkpoint max steps:", trainer_state["max_steps"])

assert torch.cuda.device_count() == 1
assert trainer.args.n_gpu == 1
assert batches_per_epoch == 1777
assert expected_total_steps == 669
assert trainer_state["max_steps"] == expected_total_steps

print("\n669-STEP SCHEDULE VERIFIED")

In [ ]:
resume_result = trainer.train(
    resume_from_checkpoint=str(CHECKPOINT)
)

print("\n=== TRAINING RESULT ===")
print("Final global step:", trainer.state.global_step)
print("Final epoch:", trainer.state.epoch)
print("Training loss:", resume_result.training_loss)

assert trainer.state.global_step == 669, (
    f"Expected 669 steps, got {trainer.state.global_step}"
)

print("\nSCENEOPS TRAINING COMPLETED")

In [ ]:
from pathlib import Path

FINAL_DIR = Path("/kaggle/working/SceneOps/final_model")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

print("Final model directory:", FINAL_DIR)
print("Files saved:")

for path in sorted(FINAL_DIR.iterdir()):
    print(path.name, f"{path.stat().st_size / 1024**2:.2f} MB")

assert (FINAL_DIR / "adapter_model.safetensors").exists()
assert (FINAL_DIR / "adapter_config.json").exists()

print("\nFINAL ADAPTER AND TOKENIZER SAVED")

In [ ]:
print("=== SCENEOPS HELD-OUT TEST EVALUATION ===")

test_metrics = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test",
)

print("\n=== FINAL TEST METRICS ===")

for key, value in test_metrics.items():
    print(f"{key}: {value}")

print("\nTest examples:", len(test_dataset))
print("Training steps:", trainer.state.global_step)

In [ ]:
import torch

test_batch = data_collator(test_rows[:5])

print("=== TEST LABEL AUDIT ===")

for i in range(5):
    ids = test_batch["input_ids"][i]
    labels = test_batch["labels"][i]
    mask = test_batch["attention_mask"][i]

    supervised = (labels != -100).sum().item()
    total = mask.sum().item()

    target_ids = labels[labels != -100]

    print(f"\nSample {i+1}")
    print("Total tokens:", total)
    print("Supervised tokens:", supervised)
    print("Masked tokens:", (labels == -100).sum().item())
    print("Target preview:")
    print(tokenizer.decode(target_ids[:100]))

    assert supervised > 0

print("\nTEST LABEL AUDIT PASSED")

In [ ]:
import torch

model.eval()

for i in [0, 100, 500]:
    sample = test_rows[i]
    messages = sample["messages"]

    prompt = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to("cuda:0")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    reference = messages[-1]["content"]

    print(f"\n{'='*65}")
    print(f"TEST SAMPLE {i}")
    print("\nREFERENCE:")
    print(reference)
    print("\nMODEL GENERATION:")
    print(generated)

print("\nQUALITATIVE GENERATION CHECK COMPLETE")

In [ ]:
import json
import re
import gc
import torch
from pathlib import Path
from difflib import SequenceMatcher
from tqdm.auto import tqdm

model.eval()

EVAL_DIR = Path("/kaggle/working/SceneOps/evaluation")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = EVAL_DIR / "test_generation_results.jsonl"

# Avoid accidentally overwriting an earlier evaluation
assert not RESULTS_FILE.exists(), (
    f"Results file already exists: {RESULTS_FILE}"
)

BATCH_SIZE = 8
MAX_NEW_TOKENS = 256

print("=== EVALUATION CONFIGURATION ===")
print("Test examples:", len(test_rows))
print("Batch size:", BATCH_SIZE)
print("Max new tokens:", MAX_NEW_TOKENS)
print("Output:", RESULTS_FILE)
print("GPU:", torch.cuda.get_device_name(0))

assert len(test_rows) == 1181
assert torch.cuda.device_count() == 1

print("\nEVALUATION SETUP READY")

In [ ]:
import json
import torch
from tqdm.auto import tqdm

tokenizer.padding_side = "left"
model.eval()

with open(RESULTS_FILE, "w", encoding="utf-8") as output_file:

    for start in tqdm(range(0, len(test_rows), 4), desc="Test generation"):

        batch = test_rows[start:start + 4]

        prompts = [
            tokenizer.apply_chat_template(
                row["messages"][:-1],
                tokenize=False,
                add_generation_prompt=True,
            )
            for row in batch
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=False,
        ).to("cuda:0")

        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_length = inputs["input_ids"].shape[1]

        for j, row in enumerate(batch):
            prediction = tokenizer.decode(
                generated[j, prompt_length:],
                skip_special_tokens=True,
            ).strip()

            reference = row["messages"][-1]["content"]

            result = {
                "test_index": start + j,
                "prediction": prediction,
                "reference": reference,
            }

            output_file.write(json.dumps(result, ensure_ascii=False) + "\n")

        output_file.flush()

        del inputs, generated
        torch.cuda.empty_cache()

print("\n=== GENERATION COMPLETE ===")
print("Generated examples:", len(test_rows))
print("Results file:", RESULTS_FILE)

In [ ]:
from pathlib import Path

BASE = Path("/kaggle/working/SceneOps")

paths = {
    "Final adapter": BASE / "final_model" / "adapter_model.safetensors",
    "Adapter config": BASE / "final_model" / "adapter_config.json",
    "Tokenizer": BASE / "final_model" / "tokenizer.json",
    "Training checkpoint": BASE / "checkpoints" / "checkpoint-223",
    "Evaluation results": BASE / "evaluation" / "test_generation_results.jsonl",
}

print("=== SCENEOPS RECOVERY CHECK ===")

for name, path in paths.items():
    if path.is_file():
        print(f"{name}: AVAILABLE ({path.stat().st_size / 1024**2:.2f} MB)")
    elif path.is_dir():
        print(f"{name}: AVAILABLE (directory)")
    else:
        print(f"{name}: NOT FOUND")

print("\nRecovery check completed.")

In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")

print("=== PERSISTENT ARTIFACT SCAN ===")

for path in sorted(INPUT.rglob("*")):
    if path.is_file() and (
        path.name.endswith(".safetensors")
        or path.name in [
            "trainer_state.json",
            "adapter_config.json",
        ]
        or path.name.endswith(".zip")
    ):
        print(
            path,
            f"| {path.stat().st_size / 1024**2:.2f} MB"
        )

print("\n=== INPUT DATASETS ===")
for path in INPUT.iterdir():
    print(path)

In [ ]:
from pathlib import Path
import shutil
import json

SOURCE = Path(
    "/kaggle/input/datasets/priyanshu4404/"
    "sceneops-training-artifacts/"
    "SceneOps_Training_Artifacts"
)

WORK = Path("/kaggle/working/SceneOps")
CHECKPOINT_SOURCE = SOURCE / "checkpoint-223"
CHECKPOINT = WORK / "checkpoints" / "checkpoint-223"
DATASET = SOURCE / "dataset"

CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT.exists():
    shutil.copytree(CHECKPOINT_SOURCE, CHECKPOINT)

with open(CHECKPOINT / "trainer_state.json") as f:
    state = json.load(f)

print("Checkpoint:", CHECKPOINT)
print("Global step:", state["global_step"])
print("Epoch:", state["epoch"])
print("Max steps:", state["max_steps"])
print("Files:", len(list(CHECKPOINT.iterdir())))

print("\nDataset available:", DATASET.exists())
print("Checkpoint copied and verified.")

In [ ]:
import shutil
from pathlib import Path

BACKUP = Path("/kaggle/working/SceneOps_recovery_backup")

shutil.copytree(
    CHECKPOINT,
    BACKUP / "checkpoint-223",
    dirs_exist_ok=True
)

shutil.copytree(
    DATASET,
    BACKUP / "dataset",
    dirs_exist_ok=True
)

print("=== RECOVERY BACKUP ===")
print("Checkpoint:", (BACKUP / "checkpoint-223").exists())
print("Dataset:", (BACKUP / "dataset").exists())

print(
    "Backup size:",
    round(
        sum(p.stat().st_size for p in BACKUP.rglob("*") if p.is_file())
        / 1024**3,
        3
    ),
    "GB"
)

print("\nORIGINAL TRAINING ARTIFACTS BACKED UP")

In [ ]:
from pathlib import Path

root = Path("/kaggle/working")

print("=== SEARCHING FOR SAVED MODEL ===")

matches = list(root.rglob("adapter_model.safetensors"))

if matches:
    for p in matches:
        print("FOUND:", p)
        print("Size:", round(p.stat().st_size / 1024**2, 2), "MB")
else:
    print("No adapter_model.safetensors found.")

print("\n=== SCENEOPS FOLDERS ===")
for p in (root / "SceneOps").rglob("*"):
    if p.is_dir():
        print("[DIR]", p)

In [ ]:
import os
from pathlib import Path
import importlib.util

print("=== PERSISTENCE CHECK ===")

# Kaggle API availability
print("Kaggle package installed:",
      importlib.util.find_spec("kaggle") is not None)

# Check authentication without displaying secrets
config = Path.home() / ".kaggle" / "kaggle.json"
env_token = bool(os.environ.get("KAGGLE_API_TOKEN"))

print("Kaggle config exists:", config.exists())
print("Kaggle API token available:", env_token)

# Verify recovery backup
backup = Path("/kaggle/working/SceneOps_recovery_backup")
checkpoint = backup / "checkpoint-223"

print("Recovery checkpoint exists:", checkpoint.exists())
print("Adapter exists:",
      (checkpoint / "adapter_model.safetensors").exists())

print("\nPERSISTENCE CHECK COMPLETE")

In [ ]:
import torch

print("=== CURRENT SESSION CHECK ===")
print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

for name in ["trainer", "model", "tokenizer", "train_dataset", "val_dataset"]:
    print(f"{name} available:", name in globals())

print("\n=== CHECKPOINT ===")
print("Path:", CHECKPOINT if "CHECKPOINT" in globals() else "Not defined")

In [ ]:
import os

# Must be set before importing PyTorch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

print("=== GPU CONFIGURATION ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
    print("BF16 supported:", torch.cuda.is_bf16_supported())

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 1
assert torch.cuda.is_bf16_supported()

print("\nSINGLE GPU CONFIGURATION VERIFIED")

In [ ]:
from pathlib import Path
import json
import shutil

SOURCE = Path(
    "/kaggle/input/datasets/priyanshu4404/"
    "sceneops-training-artifacts/SceneOps_Training_Artifacts"
)

WORK = Path("/kaggle/working/SceneOps")
CHECKPOINT_SOURCE = SOURCE / "checkpoint-223"
DATASET = SOURCE / "dataset"

CHECKPOINT = WORK / "checkpoints" / "checkpoint-223"

WORK.mkdir(parents=True, exist_ok=True)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT.exists():
    shutil.copytree(CHECKPOINT_SOURCE, CHECKPOINT)

with open(CHECKPOINT / "trainer_state.json") as f:
    state = json.load(f)

print("=== CHECKPOINT ===")
print("Path:", CHECKPOINT)
print("Global step:", state["global_step"])
print("Epoch:", state["epoch"])
print("Max steps:", state["max_steps"])

print("\n=== DATASET ===")
for name in [
    "sceneops_narration_train_v4.jsonl",
    "sceneops_narration_validation_v4.jsonl",
    "sceneops_narration_test_v4.jsonl",
]:
    path = DATASET / name
    assert path.exists(), f"Missing: {name}"
    with path.open(encoding="utf-8") as f:
        count = sum(1 for line in f if line.strip())
    print(name, ":", count)

assert state["global_step"] == 223
assert state["max_steps"] == 669

print("\nCHECKPOINT AND DATASET RESTORED")

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    prepare_model_for_kbit_training,
    PeftModel,
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)

base_model.config.use_cache = False

model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

model = PeftModel.from_pretrained(
    model,
    str(CHECKPOINT),
    is_trainable=True,
)

model.print_trainable_parameters()

print("\nGPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(
    torch.cuda.memory_allocated(0) / 1024**3, 2
), "GB")

print("\nBASE MODEL AND CHECKPOINT ADAPTER LOADED")

In [ ]:
%pip install --no-cache-dir --force-reinstall --no-deps "bitsandbytes==0.49.2"

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("BitsAndBytes:", bnb.__version__)
print("CUDA:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", torch.cuda.is_bf16_supported())

assert torch.cuda.device_count() == 1
assert torch.cuda.is_bf16_supported()
assert bnb.__version__ == "0.49.2"

print("\nENVIRONMENT VERIFIED")

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map={"": 0},
)

base_model.config.use_cache = False

model = PeftModel.from_pretrained(
    base_model,
    str(CHECKPOINT),
    is_trainable=True,
)

model.print_trainable_parameters()

print("\nGPU:", torch.cuda.get_device_name(0))
print("Memory allocated:",
      round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")

print("\nEXISTING CHECKPOINT ADAPTER LOADED")

In [ ]:
from pathlib import Path
import json

SOURCE = Path(
    "/kaggle/input/datasets/priyanshu4404/"
    "sceneops-training-artifacts/SceneOps_Training_Artifacts"
)

WORK = Path("/kaggle/working/SceneOps")
CHECKPOINT = WORK / "checkpoints" / "checkpoint-223"
DATASET = SOURCE / "dataset"

assert CHECKPOINT.exists()
assert (CHECKPOINT / "adapter_model.safetensors").exists()

with open(CHECKPOINT / "trainer_state.json") as f:
    state = json.load(f)

print("Checkpoint:", CHECKPOINT)
print("Global step:", state["global_step"])
print("Max steps:", state["max_steps"])

assert state["global_step"] == 223
assert state["max_steps"] == 669

print("\nCHECKPOINT PATH VERIFIED")

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    str(CHECKPOINT),
    is_trainable=True,
)

model.print_trainable_parameters()

print("\nAdapter loaded:", model.active_adapter)
print("Trainable:", any(p.requires_grad for p in model.parameters()))
print("GPU:", torch.cuda.get_device_name(0))

print("\nCHECKPOINT ADAPTER LOADED SUCCESSFULLY")

In [ ]:
import json
import torch

from datasets import Dataset
from dataclasses import dataclass
from trl import SFTConfig

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = load_jsonl(DATASET / "sceneops_narration_train_v4.jsonl")
val_rows = load_jsonl(DATASET / "sceneops_narration_validation_v4.jsonl")
test_rows = load_jsonl(DATASET / "sceneops_narration_test_v4.jsonl")

train_dataset = Dataset.from_list(train_rows)
val_dataset = Dataset.from_list(val_rows)
test_dataset = Dataset.from_list(test_rows)

@dataclass
class SceneOpsDataCollator:
    tokenizer: object
    max_length: int = 512

    def __call__(self, features):
        batch_ids, batch_labels = [], []

        for feature in features:
            messages = feature["messages"]

            prompt_ids = self.tokenizer.apply_chat_template(
                messages[:-1],
                tokenize=True,
                add_generation_prompt=True,
            )

            full_ids = self.tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=False,
            )

            assert full_ids[:len(prompt_ids)] == prompt_ids

            target_ids = full_ids[len(prompt_ids):]

            if len(target_ids) >= self.max_length:
                raise ValueError("Assistant target exceeds max_length")

            available_prompt = self.max_length - len(target_ids)
            prompt_ids = prompt_ids[-available_prompt:] if available_prompt else []

            batch_ids.append(prompt_ids + target_ids)
            batch_labels.append([-100] * len(prompt_ids) + target_ids)

        max_len = max(map(len, batch_ids))

        input_ids = [
            x + [self.tokenizer.pad_token_id] * (max_len - len(x))
            for x in batch_ids
        ]
        labels = [
            x + [-100] * (max_len - len(x))
            for x in batch_labels
        ]
        attention_mask = [
            [1] * len(x) + [0] * (max_len - len(x))
            for x in batch_ids
        ]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }

data_collator = SceneOpsDataCollator(tokenizer)

training_args = SFTConfig(
    output_dir=str(WORK / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    optim="adamw_torch",
    max_length=512,
    packing=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    dataloader_num_workers=0,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    report_to="none",
    run_name="SceneOps-Qwen2.5-1.5B-QLoRA",
    seed=42,
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Epochs:", training_args.num_train_epochs)
print("Max length:", training_args.max_length)

assert len(train_dataset) == 3554
assert len(val_dataset) == 1121
assert len(test_dataset) == 1181

print("\nDATASET AND CONFIGURATION RESTORED")

In [ ]:
%pip install --no-cache-dir \
    "transformers==4.57.1" \
    "trl==0.24.0" \
    "peft==0.18.0" \
    "accelerate==1.12.0" \
    "bitsandbytes==0.49.2" \
    "datasets"

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import transformers
import trl
import peft
import accelerate
import bitsandbytes
import datasets

from trl import SFTConfig, SFTTrainer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Datasets:", datasets.__version__)

assert torch.cuda.is_available(), "CUDA GPU not available"
assert torch.cuda.device_count() == 1, "Expected exactly one visible GPU"
assert torch.cuda.is_bf16_supported(), "BF16 is not supported"

print("\nGPU:", torch.cuda.get_device_name(0))
print("Visible GPUs:", torch.cuda.device_count())
print("\nENVIRONMENT VERIFIED")

In [ ]:
from pathlib import Path
import json
import shutil

SOURCE = Path(
    "/kaggle/input/datasets/priyanshu4404/"
    "sceneops-training-artifacts/SceneOps_Training_Artifacts"
)

WORK = Path("/kaggle/working/SceneOps")
CHECKPOINT_SOURCE = SOURCE / "checkpoint-223"
DATASET = SOURCE / "dataset"
CHECKPOINT = WORK / "checkpoints" / "checkpoint-223"

# Restore checkpoint only if it is not already present
WORK.mkdir(parents=True, exist_ok=True)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT.exists():
    assert CHECKPOINT_SOURCE.exists(), f"Missing source checkpoint: {CHECKPOINT_SOURCE}"
    shutil.copytree(CHECKPOINT_SOURCE, CHECKPOINT)

# Verify checkpoint state
state_file = CHECKPOINT / "trainer_state.json"
assert state_file.exists(), "trainer_state.json missing"

with open(state_file, "r") as f:
    checkpoint_state = json.load(f)

# Verify dataset files
train_file = DATASET / "sceneops_narration_train_v4.jsonl"
val_file = DATASET / "sceneops_narration_validation_v4.jsonl"
test_file = DATASET / "sceneops_narration_test_v4.jsonl"

for file in [train_file, val_file, test_file]:
    assert file.exists(), f"Dataset file missing: {file}"

print("Checkpoint:", CHECKPOINT)
print("Global step:", checkpoint_state["global_step"])
print("Checkpoint epoch:", checkpoint_state["epoch"])
print("Configured max steps:", checkpoint_state["max_steps"])

print("\nDataset files verified:")
print("Train:", train_file)
print("Validation:", val_file)
print("Test:", test_file)

assert checkpoint_state["global_step"] == 223
assert checkpoint_state["max_steps"] == 669

print("\nCHECKPOINT AND DATASET VERIFIED")

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load base model on the single T4 GPU
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map={"": 0}
)

base_model.config.use_cache = False

# Attach the existing trainable LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    str(CHECKPOINT),
    is_trainable=True
)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total_params = sum(p.numel() for p in model.parameters())

print("Base model:", MODEL_NAME)
print("Adapter:", CHECKPOINT)
print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.3f}%")
print("GPU:", torch.cuda.get_device_name(0))
print("Adapter attached:", model.active_adapter)
print("Adapter trainable:", model.peft_config["default"].inference_mode is False)

assert trainable_params > 0
assert model.peft_config["default"].inference_mode is False

print("\nMODEL AND ADAPTER READY")

In [ ]:
import json
from pathlib import Path
import peft

config_path = CHECKPOINT / "adapter_config.json"

with open(config_path, "r") as f:
    adapter_config = json.load(f)

print("Installed PEFT:", peft.__version__)
print("\nCheckpoint adapter configuration:")

for key, value in adapter_config.items():
    print(f"{key}: {value}")

print("\nLoaded adapter configuration:")
print(model.peft_config["default"])

print("\nCONFIG INSPECTION COMPLETE")

In [ ]:
%pip install --no-cache-dir --no-deps "peft==0.21.2"

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import transformers
import trl
import peft
import accelerate
import bitsandbytes
import datasets

from trl import SFTConfig, SFTTrainer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Datasets:", datasets.__version__)

assert peft.__version__ == "0.21.2", "PEFT version mismatch"
assert transformers.__version__ == "4.57.1"
assert trl.__version__ == "0.24.0"
assert torch.cuda.is_available()
assert torch.cuda.device_count() == 1
assert torch.cuda.is_bf16_supported()

print("\nGPU:", torch.cuda.get_device_name(0))
print("ENVIRONMENT VERIFIED")

In [ ]:
from pathlib import Path
import json
import shutil

SOURCE = Path(
    "/kaggle/input/datasets/priyanshu4404/"
    "sceneops-training-artifacts/SceneOps_Training_Artifacts"
)

WORK = Path("/kaggle/working/SceneOps")
CHECKPOINT_SOURCE = SOURCE / "checkpoint-223"
DATASET = SOURCE / "dataset"
CHECKPOINT = WORK / "checkpoints" / "checkpoint-223"

WORK.mkdir(parents=True, exist_ok=True)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT.exists():
    assert CHECKPOINT_SOURCE.exists(), "Original checkpoint missing"
    shutil.copytree(CHECKPOINT_SOURCE, CHECKPOINT)

with open(CHECKPOINT / "trainer_state.json") as f:
    checkpoint_state = json.load(f)

for filename in [
    "sceneops_narration_train_v4.jsonl",
    "sceneops_narration_validation_v4.jsonl",
    "sceneops_narration_test_v4.jsonl"
]:
    assert (DATASET / filename).exists(), f"Missing: {filename}"

with open(CHECKPOINT / "adapter_config.json") as f:
    adapter_config = json.load(f)

print("Checkpoint step:", checkpoint_state["global_step"])
print("Total configured steps:", checkpoint_state["max_steps"])
print("Original PEFT version:", adapter_config["peft_version"])
print("Current PEFT version:", __import__("peft").__version__)
print("Checkpoint:", CHECKPOINT)
print("Dataset:", DATASET)

assert checkpoint_state["global_step"] == 223
assert checkpoint_state["max_steps"] == 669
assert adapter_config["peft_version"] == "0.21.2"

print("\nRESTORE VERIFIED")


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

# Original 4-bit NF4 configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map={"": 0}
)

base_model.config.use_cache = False

# Attach checkpoint adapter
model = PeftModel.from_pretrained(
    base_model,
    str(CHECKPOINT),
    is_trainable=True
)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total_params = sum(p.numel() for p in model.parameters())

print("Base model:", MODEL_NAME)
print("Trainable parameters:", f"{trainable_params:,}")
print("Total parameters:", f"{total_params:,}")
print("Trainable percentage:", f"{100 * trainable_params / total_params:.3f}%")
print("Adapter:", model.active_adapter)
print("Adapter trainable:", model.peft_config["default"].inference_mode is False)
print("GPU:", torch.cuda.get_device_name(0))

assert trainable_params > 0
assert model.peft_config["default"].inference_mode is False

print("\nMODEL AND ADAPTER READY")

In [ ]:
import json

def load_jsonl_preview(path):
    with open(path, "r", encoding="utf-8") as f:
        first_line = f.readline()
    return json.loads(first_line)

train_example = load_jsonl_preview(
    DATASET / "sceneops_narration_train_v4.jsonl"
)

print("Top-level keys:", list(train_example.keys()))
print("\nFirst training example:")
print(json.dumps(train_example, indent=2, ensure_ascii=False))

print("\nTokenizer max length:", tokenizer.model_max_length)
print("Configured training sequence length: 512")

In [ ]:
import math
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = str(WORK / "checkpoints")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    optim="adamw_torch",

    max_length=512,
    packing=False,
    dataset_kwargs={"skip_prepare_dataset": True},

    fp16=False,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    remove_unused_columns=False,
    dataloader_num_workers=0,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=10,
    report_to="none",
    run_name="SceneOps-Qwen2.5-1.5B-QLoRA",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

train_dataloader = trainer.get_train_dataloader()
batches_per_epoch = len(train_dataloader)

steps_per_epoch = math.ceil(
    batches_per_epoch / training_args.gradient_accumulation_steps
)

expected_total_steps = steps_per_epoch * int(training_args.num_train_epochs)

print("Training batches per epoch:", batches_per_epoch)
print("Optimizer steps per epoch:", steps_per_epoch)
print("Expected total optimizer steps:", expected_total_steps)
print("Checkpoint global step:", checkpoint_state["global_step"])
print("Checkpoint configured max steps:", checkpoint_state["max_steps"])
print("Trainer GPU count:", trainer.args.n_gpu)
print("Training epochs:", training_args.num_train_epochs)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Sequence length:", training_args.max_length)

assert batches_per_epoch == 1777
assert steps_per_epoch == 223
assert expected_total_steps == 669
assert checkpoint_state["global_step"] == 223
assert checkpoint_state["max_steps"] == expected_total_steps
assert trainer.args.n_gpu == 1
assert trainer.state.global_step == 0

print("\nTRAINER CONFIGURATION VERIFIED — NOT STARTED")

In [ ]:
import json
import torch
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(
    DATASET / "sceneops_narration_train_v4.jsonl"
)
val_rows = load_jsonl(
    DATASET / "sceneops_narration_validation_v4.jsonl"
)
test_rows = load_jsonl(
    DATASET / "sceneops_narration_test_v4.jsonl"
)

train_dataset = Dataset.from_list(train_rows)
val_dataset = Dataset.from_list(val_rows)
test_dataset = Dataset.from_list(test_rows)

class SceneOpsDataCollator:
    def __init__(self, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, features):
        batch_input_ids = []
        batch_labels = []

        for feature in features:
            messages = feature["messages"]

            assert messages[-1]["role"] == "assistant"

            prompt_ids = self.tokenizer.apply_chat_template(
                messages[:-1],
                tokenize=True,
                add_generation_prompt=True
            )

            full_ids = self.tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=False
            )

            assert full_ids[:len(prompt_ids)] == prompt_ids, (
                "Chat template prompt/response boundary mismatch"
            )

            target_ids = full_ids[len(prompt_ids):]

            if len(target_ids) >= self.max_length:
                raise ValueError(
                    f"Assistant response exceeds max length: {len(target_ids)}"
                )

            available_prompt = self.max_length - len(target_ids)
            prompt_ids = prompt_ids[-available_prompt:] if available_prompt else []

            input_ids = prompt_ids + target_ids
            labels = [-100] * len(prompt_ids) + target_ids

            batch_input_ids.append(input_ids)
            batch_labels.append(labels)

        max_len = max(len(ids) for ids in batch_input_ids)
        pad_id = self.tokenizer.pad_token_id

        padded_inputs = []
        padded_labels = []
        attention_masks = []

        for input_ids, labels in zip(batch_input_ids, batch_labels):
            padding = max_len - len(input_ids)

            padded_inputs.append(input_ids + [pad_id] * padding)
            padded_labels.append(labels + [-100] * padding)
            attention_masks.append([1] * len(input_ids) + [0] * padding)

        return {
            "input_ids": torch.tensor(padded_inputs, dtype=torch.long),
            "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
            "labels": torch.tensor(padded_labels, dtype=torch.long)
        }

data_collator = SceneOpsDataCollator(tokenizer, max_length=512)

# Test a small batch before training
test_batch = data_collator([
    train_dataset[0],
    train_dataset[1]
])

print("Train examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Test examples:", len(test_dataset))
print("\nTest batch shapes:")

for key, value in test_batch.items():
    print(key, tuple(value.shape))

print("\nAssistant target tokens:", (test_batch["labels"] != -100).sum().item())
print("Masked prompt/padding tokens:", (test_batch["labels"] == -100).sum().item())

assert len(train_dataset) == 3554
assert len(val_dataset) == 1121
assert len(test_dataset) == 1181
assert test_batch["input_ids"].shape[1] <= 512
assert (test_batch["labels"] != -100).sum().item() > 0

print("\nDATASET AND COLLATOR VERIFIED")

In [ ]:
import time

print("Resuming from checkpoint:", CHECKPOINT)
print("Starting global step:", trainer.state.global_step)
print("Expected final global step:", 669)

start_time = time.time()

train_result = trainer.train(
    resume_from_checkpoint=str(CHECKPOINT)
)

elapsed_hours = (time.time() - start_time) / 3600

print("\nTRAINING FINISHED")
print("Final global step:", trainer.state.global_step)
print(f"Training duration: {elapsed_hours:.2f} hours")
print("Training metrics:", train_result.metrics)

assert trainer.state.global_step == 669, (
    f"Expected 669 steps, got {trainer.state.global_step}"
)

print("\nALL 669 OPTIMIZER STEPS COMPLETED")

In [ ]:
import json
import zipfile
from pathlib import Path

FINAL_DIR = WORK / "final_model"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Save trained LoRA adapter and tokenizer
model.save_pretrained(
    str(FINAL_DIR),
    safe_serialization=True
)
tokenizer.save_pretrained(str(FINAL_DIR))

# Preserve training state and metrics
trainer.state.save_to_json(str(FINAL_DIR / "trainer_state.json"))

with open(FINAL_DIR / "training_metrics.json", "w") as f:
    json.dump(train_result.metrics, f, indent=2)

# Create a portable backup
ZIP_PATH = WORK / "SceneOps_final_model.zip"

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for file in FINAL_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, file.relative_to(WORK))

# Verify saved files and archive integrity
required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer_config.json",
    "trainer_state.json",
    "training_metrics.json",
]

for filename in required_files:
    path = FINAL_DIR / filename
    assert path.exists(), f"Missing artifact: {filename}"
    print(filename, f"{path.stat().st_size / (1024**2):.2f} MB")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    assert zf.testzip() is None, "ZIP integrity check failed"
    print("\nZIP entries:", len(zf.namelist()))

print("\nFinal adapter:", FINAL_DIR)
print("Backup ZIP:", ZIP_PATH)
print("ZIP size:", f"{ZIP_PATH.stat().st_size / (1024**2):.2f} MB")
print("\nFINAL MODEL SAVED AND ZIP VERIFIED")

In [ ]:
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MERGED_DIR = WORK / "merged_model"

print("Loading original base model in FP16 on CPU...")

merge_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True
)

print("Attaching trained adapter...")

merge_model = PeftModel.from_pretrained(
    merge_base,
    str(FINAL_DIR),
    is_trainable=False
)

print("Merging LoRA weights...")

merged_model = merge_model.merge_and_unload(safe_merge=True)

print("Saving standalone merged model...")

MERGED_DIR.mkdir(parents=True, exist_ok=True)

merged_model.save_pretrained(
    str(MERGED_DIR),
    safe_serialization=True,
    max_shard_size="2GB"
)

tokenizer.save_pretrained(str(MERGED_DIR))

# Verify essential files
weight_files = list(MERGED_DIR.glob("*.safetensors"))
assert weight_files, "Merged model weights were not saved"
assert (MERGED_DIR / "config.json").exists(), "Missing model config"

total_size = sum(f.stat().st_size for f in weight_files)

print("\nMerged model:", MERGED_DIR)
print("Weight shards:", len(weight_files))
print(f"Total weight size: {total_size / (1024**3):.2f} GB")
print("MERGED MODEL VERIFIED")

In [ ]:
%pip uninstall -y torchao

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import peft
import transformers
import importlib.util
import zipfile
from pathlib import Path

WORK = Path("/kaggle/working/SceneOps")
FINAL_DIR = WORK / "final_model"
ZIP_PATH = WORK / "SceneOps_final_model.zip"

print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)
print("TorchAO installed:", importlib.util.find_spec("torchao") is not None)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 1
assert peft.__version__ == "0.21.2"
assert importlib.util.find_spec("torchao") is None, "TorchAO still detected"

assert ZIP_PATH.exists(), "Final model ZIP missing"
assert (FINAL_DIR / "adapter_model.safetensors").exists()

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    assert zf.testzip() is None, "ZIP integrity failed"

print("\nGPU:", torch.cuda.get_device_name(0))
print("Adapter size:", round(
    (FINAL_DIR / "adapter_model.safetensors").stat().st_size / 1024**2, 2
), "MB")
print("ZIP integrity: PASS")
print("\nREADY FOR MODEL MERGING")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from pathlib import Path

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MERGED_DIR = WORK / "merged_model"

assert (FINAL_DIR / "adapter_model.safetensors").exists()

print("1. Loading base model in FP16 on CPU...")

merge_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True
)

print("2. Loading trained LoRA adapter...")

merge_model = PeftModel.from_pretrained(
    merge_base,
    str(FINAL_DIR),
    is_trainable=False
)

print("3. Merging adapter weights...")

merged_model = merge_model.merge_and_unload(safe_merge=True)
merged_model.eval()

print("4. Saving merged model...")

MERGED_DIR.mkdir(parents=True, exist_ok=True)

merged_model.save_pretrained(
    str(MERGED_DIR),
    safe_serialization=True,
    max_shard_size="2GB"
)

merge_tokenizer = AutoTokenizer.from_pretrained(str(FINAL_DIR))
merge_tokenizer.save_pretrained(str(MERGED_DIR))

# Verify output
weight_files = list(MERGED_DIR.glob("*.safetensors"))
assert weight_files, "Merged weights missing"
assert (MERGED_DIR / "config.json").exists()
assert (MERGED_DIR / "tokenizer_config.json").exists()

total_size = sum(f.stat().st_size for f in weight_files)

print("\nMerged directory:", MERGED_DIR)
print("Weight files:", [f.name for f in weight_files])
print(f"Total weights: {total_size / (1024**3):.2f} GB")
print("\nMERGED MODEL SAVED AND VERIFIED")

In [ ]:
import importlib.util
import shutil
import torch
from pathlib import Path

packages = [
    "autoawq",
    "awq",
    "llama_cpp",
    "ray",
    "vllm",
    "huggingface_hub",
    "safetensors"
]

print("=== PACKAGE AVAILABILITY ===")
for package in packages:
    print(f"{package}: {importlib.util.find_spec(package) is not None}")

print("\n=== EXECUTABLES ===")
for executable in ["llama-quantize", "llama-cli", "git", "cmake"]:
    print(f"{executable}: {shutil.which(executable)}")

print("\n=== GPU ===")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM total:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")
print("VRAM free:", round(
    torch.cuda.mem_get_info()[0] / 1024**3, 2
), "GB")

print("\n=== DISK ===")
usage = shutil.disk_usage("/kaggle/working")
print("Free space:", round(usage.free / 1024**3, 2), "GB")

print("\n=== MERGED MODEL ===")
print("Exists:", MERGED_DIR.exists())
print("Size:", round(
    sum(p.stat().st_size for p in MERGED_DIR.rglob("*") if p.is_file()) / 1024**3,
    2
), "GB")

In [ ]:
%pip install --no-cache-dir llmcompressor

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import transformers
import accelerate
import ray
import llmcompressor
import compressed_tensors

from pathlib import Path

WORK = Path("/kaggle/working/SceneOps")
MERGED_DIR = WORK / "merged_model"
FINAL_DIR = WORK / "final_model"
ZIP_PATH = WORK / "SceneOps_final_model.zip"

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Ray:", ray.__version__)
print("LLM Compressor:", llmcompressor.__version__)
print("Compressed Tensors:", compressed_tensors.__version__)

print("\nGPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

print("\nMerged model exists:", MERGED_DIR.exists())
print("Final adapter exists:", FINAL_DIR.exists())
print("Backup ZIP exists:", ZIP_PATH.exists())

assert torch.cuda.is_available()
assert MERGED_DIR.exists()
assert (MERGED_DIR / "config.json").exists()
assert ZIP_PATH.exists()

print("\nQUANTIZATION ENVIRONMENT VERIFIED")

PyTorch: 2.11.0+cu130
Transformers: 5.21.0
Accelerate: 1.16.0
Ray: 2.59.0
LLM Compressor: 0.15.0
Compressed Tensors: 0.20.1

GPU: Tesla T4
CUDA: True

Merged model exists: True
Final adapter exists: True
Backup ZIP exists: True

QUANTIZATION ENVIRONMENT VERIFIED


In [2]:
import json
import inspect
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.transform.awq import AWQModifier
from llmcompressor.modifiers.quantization import QuantizationModifier

SOURCE = Path(
    "/kaggle/input/datasets/priyanshu4404/"
    "sceneops-training-artifacts/SceneOps_Training_Artifacts"
)
DATASET_DIR = SOURCE / "dataset"
MERGED_DIR = Path("/kaggle/working/SceneOps/merged_model")
AWQ_DIR = Path("/kaggle/working/SceneOps/awq_w4a16")

assert MERGED_DIR.exists()
assert (MERGED_DIR / "config.json").exists()

tokenizer = AutoTokenizer.from_pretrained(str(MERGED_DIR))

# Build representative calibration prompts from training data.
calibration_rows = []

with open(DATASET_DIR / "sceneops_narration_train_v4.jsonl", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        messages = row["messages"]

        # Use the complete authorized conversation as calibration text.
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        calibration_rows.append({"text": text})

        if len(calibration_rows) >= 128:
            break

calibration_dataset = Dataset.from_list(calibration_rows)

print("Calibration samples:", len(calibration_dataset))
print("Sample text preview:")
print(calibration_dataset[0]["text"][:500])

print("\nAWQModifier signature:")
print(inspect.signature(AWQModifier))

print("\nQuantizationModifier signature:")
print(inspect.signature(QuantizationModifier))

print("\noneshot signature:")
print(inspect.signature(oneshot))

print("\nAWQ output directory:", AWQ_DIR)
print("CALIBRATION DATA READY")

Calibration samples: 128
Sample text preview:
<|im_start|>system
You narrate a synthetic educational mass-casualty training scene. Use only learner-recorded observations supplied in the user message. Do not infer causes, injuries, symptoms, diagnoses, triage categories, or events. Do not reveal hidden simulator state.<|im_end|>
<|im_start|>user
Summarize the learner-recorded observations for patient P001 at simulation second 600: respiratory rate: 22; radial pulse: present; capillary refill: 0.8; mental status: confused; ambulatory status: 

AWQModifier signature:
(*, requires_calibration_data: bool = True, index: int | None = None, group: str | None = None, start: float | None = None, end: float | None = None, update: float | None = None, initialized_: bool = False, finalized_: bool = False, started_: bool = False, ended_: bool = False, mappings: list[llmcompressor.modifiers.transform.awq.mappings.AWQMapping] | None = None, offload_device: torch.device | None | llmcompressor.sentinel.

In [3]:
import torch
from transformers import AutoModelForCausalLM
from llmcompressor import oneshot
from llmcompressor.modifiers.transform.awq import AWQModifier
from llmcompressor.modifiers.quantization import QuantizationModifier

print("Loading merged model for AWQ...")

awq_model = AutoModelForCausalLM.from_pretrained(
    str(MERGED_DIR),
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

recipe = [
    AWQModifier(duo_scaling=True),
    QuantizationModifier(
        targets=["Linear"],
        scheme="W4A16_ASYM",
        ignore=["lm_head"]
    )
]

print("Starting AWQ calibration and quantization...")
print("Calibration samples:", len(calibration_dataset))

oneshot(
    model=awq_model,
    tokenizer=tokenizer,
    dataset=calibration_dataset,
    recipe=recipe,
    max_seq_length=512,
    num_calibration_samples=128,
    batch_size=1,
    text_column="text",
    output_dir=str(AWQ_DIR)
)

print("\nAWQ PROCESS FINISHED")

# Verify artifacts
assert AWQ_DIR.exists()
assert (AWQ_DIR / "config.json").exists()

awq_files = list(AWQ_DIR.rglob("*.safetensors"))
assert awq_files, "AWQ weight files missing"

awq_size = sum(f.stat().st_size for f in awq_files)

print("AWQ directory:", AWQ_DIR)
print("Weight files:", len(awq_files))
print(f"Quantized weight size: {awq_size / (1024**3):.2f} GB")
print("\nAWQ ARTIFACTS VERIFIED")

Loading merged model for AWQ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Starting AWQ calibration and quantization...
Calibration samples: 128
2026-11-19T11:55:49.9988 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.


Tokenizing:   0%|          | 0/128 [00:00<?, ? examples/s]

2026-11-19T11:55:52.5620 | reset | INFO - Compression lifecycle reset
2026-11-19T11:55:52.5676 | from_modifiers | INFO - Creating recipe from modifiers
2026-11-19T11:55:52.5685 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...


Applying quantization config: 100%|██████████| 196/196 [00:00<00:00, 4378.91it/s]

... (9 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-19T12:01:46.9571 | resave_config | INFO - Resaved original config with patched fields to /kaggle/working/SceneOps/awq_w4a16/config.json


Dispatching model: 100%|██████████| 371/371 [00:00<00:00, 14699.06it/s]


AWQ PROCESS FINISHED
AWQ directory: /kaggle/working/SceneOps/awq_w4a16
Weight files: 1
Quantized weight size: 1.07 GB

AWQ ARTIFACTS VERIFIED


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading quantized AWQ model...")

awq_tokenizer = AutoTokenizer.from_pretrained(str(AWQ_DIR))

awq_test_model = AutoModelForCausalLM.from_pretrained(
    str(AWQ_DIR),
    device_map="auto",
    dtype="auto"
)
awq_test_model.eval()

# Use a real held-out style example as the inference prompt
sample_messages = calibration_rows[0]["text"]

# Rebuild prompt from the original training example
import json

with open(DATASET_DIR / "sceneops_narration_train_v4.jsonl", encoding="utf-8") as f:
    sample = json.loads(f.readline())

prompt = awq_tokenizer.apply_chat_template(
    sample["messages"][:-1],
    tokenize=False,
    add_generation_prompt=True
)

inputs = awq_tokenizer(
    prompt,
    return_tensors="pt"
).to(awq_test_model.device)

print("Generating response...")

with torch.inference_mode():
    output = awq_test_model.generate(
        **inputs,
        max_new_tokens=160,
        do_sample=False,
        pad_token_id=awq_tokenizer.pad_token_id
    )

generated = awq_tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("\nAWQ GENERATED NARRATION:\n")
print(generated)

assert len(generated.strip()) > 20
print("\nAWQ LOAD AND GENERATION TEST PASSED")

Loading quantized AWQ model...


Compressing model: 100%|██████████| 196/196 [00:00<00:00, 7187.87it/s]


Loading weights:   0%|          | 0/926 [00:00<?, ?it/s]

Generating response...


Decompressing model: 100%|██████████| 196/196 [00:00<00:00, 710.87it/s]



AWQ GENERATED NARRATION:

At simulation second 600, the following observations were recorded for patient P001. The recorded respiratory rate was 22 breaths per minute. A radial pulse was present. Capillary refill was recorded as 0.8 seconds. The recorded mental status was confused. The patient was ambulatory. No visible major hemorrhage was recorded. The patient followed commands.

AWQ LOAD AND GENERATION TEST PASSED


In [5]:
import subprocess
import os
from pathlib import Path

LLAMA_DIR = Path("/kaggle/working/llama.cpp")

if not LLAMA_DIR.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/ggml-org/llama.cpp.git",
            str(LLAMA_DIR)
        ],
        check=True
    )

# Build CPU quantization executable
subprocess.run(
    [
        "cmake", "-S", str(LLAMA_DIR), "-B",
        str(LLAMA_DIR / "build"),
        "-DGGML_CUDA=OFF",
        "-DCMAKE_BUILD_TYPE=Release"
    ],
    check=True
)

subprocess.run(
    [
        "cmake", "--build", str(LLAMA_DIR / "build"),
        "--config", "Release",
        "--target", "llama-quantize",
        "-j", "2"
    ],
    check=True
)

QUANTIZE_BIN = LLAMA_DIR / "build" / "bin" / "llama-quantize"
CONVERT_SCRIPT = LLAMA_DIR / "convert_hf_to_gguf.py"

assert QUANTIZE_BIN.exists(), "llama-quantize binary missing"
assert CONVERT_SCRIPT.exists(), "HF-to-GGUF conversion script missing"

print("llama.cpp:", LLAMA_DIR)
print("Quantizer:", QUANTIZE_BIN)
print("Converter:", CONVERT_SCRIPT)
print("\nLLAMA.CPP BUILD VERIFIED")

Cloning into '/kaggle/working/llama.cpp'...


-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 1.0.0-dev
-- Found Git: /usr/bin/git (found version "2.43.0")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE


CMAKE_BUILD_TYPE=Release


-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 1.0.0
-- ggml commit:  f8caa5e
-- Found OpenSSL: /usr/lib/x86_64-linux-gnu/libcrypto.so (found version "3.0.13")
-- Performing Test OPENSSL_VERSION_SUPPORTED
-- Performing Test OPENSSL_VERSION_SUPPORTED - Success
-- OpenSSL found: 3.0.13
-- Generating embedded license file for target: llama-app
-- Configuring done (1.7s)
-- Generating done (0.4s)
-- Build files have been written to: /kaggle/working/llama.cpp/build
[  2%] Building CXX object common/CMakeFiles/llama-common-base.dir/build-info.cpp.o
[  2%] Building C object ggml/src/CMakeFiles/ggml-base.dir

In [6]:
import subprocess
import sys
import shutil
from pathlib import Path

LLAMA_DIR = Path("/kaggle/working/llama.cpp")
CONVERT_SCRIPT = LLAMA_DIR / "convert_hf_to_gguf.py"
QUANTIZE_BIN = LLAMA_DIR / "build/bin/llama-quantize"

GGUF_DIR = WORK / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)

F16_FILE = GGUF_DIR / "sceneops-f16.gguf"
Q4_FILE = GGUF_DIR / "sceneops-q4_k_m.gguf"

assert MERGED_DIR.exists()
assert CONVERT_SCRIPT.exists()
assert QUANTIZE_BIN.exists()

free_gb = shutil.disk_usage("/kaggle/working").free / (1024**3)
print(f"Available disk space: {free_gb:.2f} GB")
assert free_gb > 5, "Insufficient disk space for GGUF conversion"

# Stage 1: Hugging Face FP16 -> GGUF F16
if not F16_FILE.exists():
    print("\nStage 1: Converting merged model to GGUF F16...")

    subprocess.run(
        [
            sys.executable,
            str(CONVERT_SCRIPT),
            str(MERGED_DIR),
            "--outfile", str(F16_FILE),
            "--outtype", "f16"
        ],
        check=True
    )
else:
    print("F16 GGUF already exists; skipping conversion.")

assert F16_FILE.exists() and F16_FILE.stat().st_size > 0

print(f"F16 GGUF size: {F16_FILE.stat().st_size / (1024**3):.2f} GB")

# Stage 2: GGUF F16 -> Q4_K_M
if not Q4_FILE.exists():
    print("\nStage 2: Quantizing to Q4_K_M...")

    subprocess.run(
        [
            str(QUANTIZE_BIN),
            str(F16_FILE),
            str(Q4_FILE),
            "Q4_K_M"
        ],
        check=True
    )
else:
    print("Q4_K_M GGUF already exists; skipping quantization.")

assert Q4_FILE.exists() and Q4_FILE.stat().st_size > 0

print("\nF16 GGUF:", F16_FILE)
print("Q4_K_M GGUF:", Q4_FILE)
print(f"Q4_K_M size: {Q4_FILE.stat().st_size / (1024**3):.2f} GB")
print("\nGGUF CONVERSION AND QUANTIZATION VERIFIED")

Available disk space: 14.53 GB

Stage 1: Converting merged model to GGUF F16...


INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.f

F16 GGUF size: 2.88 GB

Stage 2: Quantizing to Q4_K_M...


version: 1.0.0-dev (build 1, commit f8caa5e)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/kaggle/working/SceneOps/gguf/sceneops-f16.gguf' to '/kaggle/working/SceneOps/gguf/sceneops-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 31 key-value pairs and 338 tensors from /kaggle/working/SceneOps/gguf/sceneops-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv 


llama_quantize: quantize time = 71178.78 ms
llama_quantize:    total time = 71178.78 ms

F16 GGUF: /kaggle/working/SceneOps/gguf/sceneops-f16.gguf
Q4_K_M GGUF: /kaggle/working/SceneOps/gguf/sceneops-q4_k_m.gguf
Q4_K_M size: 0.92 GB

GGUF CONVERSION AND QUANTIZATION VERIFIED


size =    26.25 MiB ->     7.38 MiB
llama_model_quantize_impl: model size  =  2944.68 MiB (16.00 BPW)
llama_model_quantize_impl: quant size  =   934.69 MiB (5.08 BPW)


In [7]:
from pathlib import Path
import subprocess

WORK = Path("/kaggle/working/SceneOps")
LLAMA_CPP = Path("/kaggle/working/llama.cpp")
MODEL = WORK / "gguf" / "sceneops-q4_k_m.gguf"
BUILD = LLAMA_CPP / "build"
CLI = BUILD / "bin" / "llama-cli"

assert MODEL.exists(), f"GGUF model missing: {MODEL}"

# Build llama-cli only if it is not already available
if not CLI.exists():
    subprocess.run(
        ["cmake", "--build", str(BUILD), "--target", "llama-cli", "-j2"],
        check=True
    )

assert CLI.exists(), "llama-cli binary was not created"

prompt = (
    "You are a medical mass-casualty simulation narrator. "
    "Describe only these recorded observations; do not infer a diagnosis "
    "or make triage decisions.\n"
    "Patient P099, simulation second 120: respiratory rate 24 breaths per minute, "
    "radial pulse present, capillary refill 2.0 seconds, mental status confused, "
    "ambulatory false, no visible major hemorrhage, follows commands.\n"
    "Narration:"
)

result = subprocess.run(
    [
        str(CLI),
        "-m", str(MODEL),
        "-p", prompt,
        "-n", "120",
        "--temp", "0",
        "--no-display-prompt",
        "-c", "1024"
    ],
    capture_output=True,
    text=True,
    timeout=300
)

print("RETURN CODE:", result.returncode)
print("\nGGUF GENERATED OUTPUT:\n")
print(result.stdout[-5000:])
if result.stderr:
    print("\nSTDERR:\n", result.stderr[-2000:])

assert result.returncode == 0, "GGUF inference failed"
assert result.stdout.strip(), "GGUF returned empty output"

print("\nGGUF INFERENCE SMOKE TEST PASSED")

[  2%] Building CXX object vendor/hash/CMakeFiles/vendor-hash.dir/hash.cpp.o
[  6%] Built target ggml-base
[  6%] Built target cpp-httplib
[  8%] Built target llama-common-base
[  8%] Provisioning UI assets
-- UI: downloading from b1: https://huggingface.co/buckets/ggml-org/llama-ui/resolve/b1/dist.tar.gz
-- UI: download dist.tar.gz.sha256 from b1 failed: "HTTP response code said error"
-- UI: downloading from latest: https://huggingface.co/buckets/ggml-org/llama-ui/resolve/latest/dist.tar.gz
[  8%] Building C object vendor/hash/CMakeFiles/vendor-hash.dir/xxhash/xxhash.c.o
-- UI: archive verified and extracted
-- UI: HF download succeeded, stamp updated (ggml-org/llama-ui|b1, resolved: latest)
[  8%] Building CXX object vendor/hash/CMakeFiles/vendor-hash.dir/sha1/sha1.c.o
-- UI: gzip compression applied (/kaggle/working/llama.cpp/build/ui/apps/chat/ui-gzip/_gzip)
[  8%] Building C object vendor/hash/CMakeFiles/vendor-hash.dir/sha256/sha256.c.o
[ 11%] Linking CXX static library libvendo

In [8]:
import importlib.util
import os
import ray

print("Ray:", ray.__version__)
print("Ray Serve available:", importlib.util.find_spec("ray.serve") is not None)
print("FastAPI available:", importlib.util.find_spec("fastapi") is not None)
print("Requests available:", importlib.util.find_spec("requests") is not None)
print("W&B available:", importlib.util.find_spec("wandb") is not None)
print("W&B API key configured:", bool(os.environ.get("WANDB_API_KEY")))

try:
    from ray import serve
    print("Ray Serve import: SUCCESS")
except Exception as e:
    print("Ray Serve import: FAILED")
    print(type(e).__name__, str(e))

try:
    import fastapi
    print("FastAPI version:", fastapi.__version__)
except Exception as e:
    print("FastAPI check failed:", repr(e))

Ray: 2.59.0
Ray Serve available: True
FastAPI available: True
Requests available: True
W&B available: True
W&B API key configured: False
Ray Serve import: SUCCESS
FastAPI version: 0.141.1


In [9]:
from pathlib import Path
import requests
import ray
import torch

from ray import serve
from fastapi import FastAPI, HTTPException
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_DIR = "/kaggle/working/SceneOps/awq_w4a16"
assert Path(MODEL_DIR).exists(), "AWQ model directory not found"

# Start Ray
if not ray.is_initialized():
    ray.init(ignore_reinit_error=True, include_dashboard=False)

app = FastAPI(title="SceneOps Narration API", version="1.0")

@serve.deployment(
    num_replicas=1,
    ray_actor_options={"num_gpus": 1}
)
@serve.ingress(app)
class SceneOpsNarrator:

    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            MODEL_DIR,
            trust_remote_code=True
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            MODEL_DIR,
            torch_dtype="auto",
            device_map="auto",
            trust_remote_code=True
        )
        self.model.eval()

    @app.get("/health")
    async def health(self):
        return {"status": "healthy", "model": "SceneOps-AWQ"}

    @app.post("/generate")
    async def generate(self, payload: dict):
        prompt = payload.get("prompt", "").strip()

        if not prompt:
            raise HTTPException(status_code=400, detail="Prompt is required")

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt"
        ).to("cuda")

        with torch.inference_mode():
            output = self.model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id
            )

        generated = self.tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        return {"generated_text": generated}

# Deploy
deployment = serve.run(
    SceneOpsNarrator.bind(),
    name="sceneops-narrator",
    route_prefix="/"
)

# Test health endpoint
health = requests.get("http://127.0.0.1:8000/health", timeout=60)
print("HEALTH:", health.status_code, health.text)
assert health.status_code == 200

# Test generation endpoint
test_prompt = (
    "Describe only the recorded observations. Do not infer diagnosis "
    "or make triage decisions.\n"
    "Patient P099, simulation second 120: respiratory rate 24 breaths "
    "per minute, radial pulse present, capillary refill 2.0 seconds, "
    "mental status confused, not ambulatory, no visible major "
    "hemorrhage, follows commands.\nNarration:"
)

response = requests.post(
    "http://127.0.0.1:8000/generate",
    json={"prompt": test_prompt},
    timeout=180
)

print("\nHTTP STATUS:", response.status_code)
print("\nGENERATED NARRATION:\n", response.text)

assert response.status_code == 200
assert response.json().get("generated_text", "").strip()

print("\nRAY SERVE DEPLOYMENT AND API TEST PASSED")

2026-11-19 12:21:06,614	WARNING authentication_token_setup.py:85 -- Token authentication is enabled for this Ray cluster. Set RAY_AUTH_MODE=disabled to opt out. For more information, see https://docs.ray.io/en/latest/ray-security/token-auth.html
2026-11-19 12:21:06,616	INFO authentication_token_setup.py:50 -- Generated new authentication token and saved to /root/.ray/auth_token
2026-11-19 12:21:14,365	INFO worker.py:2033 -- Started a local Ray instance.
(ProxyActor pid=5564) INFO 2026-11-19 12:21:20,248 proxy 172.19.2.2 -- Proxy starting on node db14b7cef5cd0a19a9a74d3aee34cb981c837682e61f70fc03552da5 (HTTP port: 8000).
INFO 2026-11-19 12:21:20,433 serve 557 -- Started Serve in namespace "serve".
(ProxyActor pid=5564) INFO 2026-11-19 12:21:20,428 proxy 172.19.2.2 -- Got updated endpoints: {}.
(ServeController pid=5563) INFO 2026-11-19 12:21:20,489 controller 5563 -- Deploying new version of Deployment(name='SceneOpsNarrator', app='sceneops-narrator') (initial target replicas: 1).
(Prox

HEALTH: 200 {"status":"healthy","model":"SceneOps-AWQ"}


Decompressing model: 100%|██████████| 196/196 [00:00<00:00, 573.61it/s]



HTTP STATUS: 200

GENERATED NARRATION:
 {"generated_text":" At simulation second 120, the following observations were recorded for patient P099. The recorded respiratory rate was 24 breaths per minute. A radial pulse was present. Capillary refill was recorded as 2.0 seconds. The patient was not ambulatory. No visible major hemorrhage was recorded. The patient followed commands. To provide a more detailed and accurate answer, additional information such as the patient's name, age, gender, and any other relevant medical history would be required. However, based on the provided information, the following observations were recorded at simulation second 120 for patient P09"}

RAY SERVE DEPLOYMENT AND API TEST PASSED


(ServeReplica:sceneops-narrator:SceneOpsNarrator pid=5566) INFO 2026-11-19 12:21:40,887 sceneops-narrator_SceneOpsNarrator 6qjbpt3q ca60c5f6-fcfe-4ce2-9c08-c52ddfc727a1 -- POST /generate 200 8270.8ms


In [10]:
from ray import serve
from fastapi import FastAPI, HTTPException
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import requests

MODEL_DIR = "/kaggle/working/SceneOps/awq_w4a16"

# Replace the previous deployment
serve.delete("sceneops-narrator")

app_v2 = FastAPI(title="SceneOps Narration API", version="1.1")

@serve.deployment(
    num_replicas=1,
    ray_actor_options={"num_gpus": 1}
)
@serve.ingress(app_v2)
class SceneOpsNarratorV2:

    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            MODEL_DIR,
            trust_remote_code=True
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            MODEL_DIR,
            torch_dtype="auto",
            device_map="auto",
            trust_remote_code=True
        )
        self.model.eval()

    @app_v2.get("/health")
    async def health(self):
        return {"status": "healthy", "model": "SceneOps-AWQ-v2"}

    @app_v2.post("/generate")
    async def generate(self, payload: dict):
        user_prompt = payload.get("prompt", "").strip()

        if not user_prompt:
            raise HTTPException(status_code=400, detail="Prompt is required")

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a narration engine for a medical mass-casualty "
                    "simulation. Convert only the explicitly provided recorded "
                    "observations into concise factual narration. Preserve every "
                    "provided observation, including mental status and mobility. "
                    "Never infer missing values, diagnoses, causes, triage "
                    "categories, or treatment decisions. Never request additional "
                    "patient information. Output narration only."
                )
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]

        inputs = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to("cuda")

        with torch.inference_mode():
            output = self.model.generate(
                **inputs,
                max_new_tokens=180,
                do_sample=False,
                repetition_penalty=1.05,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        generated = self.tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        return {"generated_text": generated}

serve.run(
    SceneOpsNarratorV2.bind(),
    name="sceneops-narrator",
    route_prefix="/"
)

health = requests.get("http://127.0.0.1:8000/health", timeout=60)
print("HEALTH:", health.status_code, health.text)
assert health.status_code == 200

test_prompt = (
    "Narrate these recorded observations for patient P099 at simulation "
    "second 120: respiratory rate 24 breaths per minute; radial pulse present; "
    "capillary refill 2.0 seconds; mental status confused; not ambulatory; "
    "no visible major hemorrhage; follows commands."
)

response = requests.post(
    "http://127.0.0.1:8000/generate",
    json={"prompt": test_prompt},
    timeout=180
)

print("\nHTTP STATUS:", response.status_code)
generated_text = response.json().get("generated_text", "")
print("\nGENERATED NARRATION:\n", generated_text)

assert response.status_code == 200
assert generated_text, "Empty generation"

required_terms = [
    "24", "2.0", "confused", "not ambulatory",
    "hemorrhage", "followed commands"
]

print("\nOBSERVATION COVERAGE:")
for term in required_terms:
    print(f"{term}: {term.lower() in generated_text.lower()}")

print("\nCHAT-TEMPLATE RAY SERVE TEST COMPLETED")

INFO 2026-11-19 12:23:23,623 serve 557 -- Deleting app ['sceneops-narrator']
(ProxyActor pid=5564) INFO 2026-11-19 12:23:23,719 proxy 172.19.2.2 -- Got updated endpoints: {}.
(ServeController pid=5563) INFO 2026-11-19 12:23:23,820 controller 5563 -- Removing 1 replica from Deployment(name='SceneOpsNarrator', app='sceneops-narrator').
(ServeController pid=5563) INFO 2026-11-19 12:23:25,873 controller 5563 -- Replica(id='6qjbpt3q', deployment='SceneOpsNarrator', app='sceneops-narrator') is stopped.
INFO 2026-11-19 12:23:26,710 serve 557 -- Connecting to existing Serve app in namespace "serve". New http_options/controller_options will not be applied.
(ServeController pid=5563) INFO 2026-11-19 12:23:26,798 controller 5563 -- Deploying new version of Deployment(name='SceneOpsNarratorV2', app='sceneops-narrator') (initial target replicas: 1).
(ProxyActor pid=5564) INFO 2026-11-19 12:23:26,803 proxy 172.19.2.2 -- Got updated endpoints: {Deployment(name='SceneOpsNarratorV2', app='sceneops-narr

HEALTH: 200 {"status":"healthy","model":"SceneOps-AWQ-v2"}


Decompressing model: 100%|██████████| 196/196 [00:00<00:00, 596.60it/s]



HTTP STATUS: 200

GENERATED NARRATION:
 At simulation second 120, the following observations were recorded for patient P099. The recorded respiratory rate was 24 breaths per minute. A radial pulse was present. Capillary refill was recorded as 2.0 seconds. The recorded mental status was confused. The patient was not ambulatory. No visible major hemorrhage was recorded. The patient followed commands.

OBSERVATION COVERAGE:
24: True
2.0: True
confused: True
not ambulatory: True
hemorrhage: True
followed commands: True

CHAT-TEMPLATE RAY SERVE TEST COMPLETED


(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:23:44,391 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 92653968-ec27-4980-9a25-4f64b1e5ac81 -- POST /generate 200 5511.6ms


In [11]:
from pathlib import Path
import json

WORK = Path("/kaggle/working/SceneOps")

files_to_check = [
    WORK / "sceneops_narration_test_v4.jsonl",
    WORK / "final_model" / "training_metrics.json",
    WORK / "final_model" / "trainer_state.json",
]

for path in files_to_check:
    print(f"\n{'='*70}\n{path}")
    print("Exists:", path.exists())
    if not path.exists():
        continue

    if path.suffix == ".jsonl":
        with path.open("r", encoding="utf-8") as f:
            rows = [json.loads(line) for line in f if line.strip()]
        print("Rows:", len(rows))
        if rows:
            print("First row keys:", list(rows[0].keys()))
            print("First row sample:")
            print(json.dumps(rows[0], indent=2)[:3000])

    else:
        data = json.loads(path.read_text(encoding="utf-8"))
        if path.name == "trainer_state.json":
            print("Global step:", data.get("global_step"))
            print("Epoch:", data.get("epoch"))
            print("Best metric:", data.get("best_metric"))
            print("Log history tail:")
            print(json.dumps(data.get("log_history", [])[-5:], indent=2)[:3000])
        else:
            print(json.dumps(data, indent=2)[:4000])


/kaggle/working/SceneOps/sceneops_narration_test_v4.jsonl
Exists: False

/kaggle/working/SceneOps/final_model/training_metrics.json
Exists: True
{
  "train_runtime": 12439.4168,
  "train_samples_per_second": 0.857,
  "train_steps_per_second": 0.054,
  "total_flos": 2.4318605381935104e+16,
  "train_loss": 5.6689645031290285e-06,
  "epoch": 3.0
}

/kaggle/working/SceneOps/final_model/trainer_state.json
Exists: True
Global step: 669
Epoch: 3.0
Best metric: 6.620097792620072e-06
Log history tail:
[
  {
    "entropy": 0.72958984375,
    "epoch": 2.8733821046707932,
    "grad_norm": 7.953744352562353e-05,
    "learning_rate": 1.0525906320453916e-06,
    "loss": 0.0,
    "mean_token_accuracy": 1.0,
    "num_tokens": 1746042.0,
    "step": 640
  },
  {
    "entropy": 0.727197265625,
    "epoch": 2.9184018007878447,
    "grad_norm": 8.803738455753773e-05,
    "learning_rate": 4.682751452394052e-07,
    "loss": 0.0,
    "mean_token_accuracy": 1.0,
    "num_tokens": 1787877.0,
    "step": 650
  

In [12]:
from pathlib import Path

SEARCH_ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

patterns = [
    "*narration*train*.jsonl",
    "*narration*validation*.jsonl",
    "*narration*test*.jsonl",
    "*.jsonl",
]

for root in SEARCH_ROOTS:
    print(f"\n{'='*65}\nSEARCHING: {root}")
    if not root.exists():
        print("Directory not found")
        continue

    for pattern in patterns:
        matches = list(root.rglob(pattern))
        print(f"\n{pattern} — {len(matches)} match(es)")
        for path in matches[:30]:
            print(f"{path}  ({path.stat().st_size / (1024*1024):.2f} MB)")

print("\n\nPROJECT DIRECTORY:")
project = Path("/kaggle/working/SceneOps")
if project.exists():
    for path in sorted(project.iterdir()):
        kind = "DIR " if path.is_dir() else "FILE"
        size = path.stat().st_size / (1024*1024) if path.is_file() else 0
        print(f"{kind}  {path.name}  {size:.2f} MB")


SEARCHING: /kaggle/working

*narration*train*.jsonl — 1 match(es)
/kaggle/working/SceneOps_recovery_backup/dataset/sceneops_narration_train_v4.jsonl  (4.32 MB)

*narration*validation*.jsonl — 1 match(es)
/kaggle/working/SceneOps_recovery_backup/dataset/sceneops_narration_validation_v4.jsonl  (1.37 MB)

*narration*test*.jsonl — 1 match(es)
/kaggle/working/SceneOps_recovery_backup/dataset/sceneops_narration_test_v4.jsonl  (1.44 MB)

*.jsonl — 3 match(es)
/kaggle/working/SceneOps_recovery_backup/dataset/sceneops_narration_test_v4.jsonl  (1.44 MB)
/kaggle/working/SceneOps_recovery_backup/dataset/sceneops_narration_train_v4.jsonl  (4.32 MB)
/kaggle/working/SceneOps_recovery_backup/dataset/sceneops_narration_validation_v4.jsonl  (1.37 MB)

SEARCHING: /kaggle/input

*narration*train*.jsonl — 1 match(es)
/kaggle/input/datasets/priyanshu4404/sceneops-training-artifacts/SceneOps_Training_Artifacts/dataset/sceneops_narration_train_v4.jsonl  (4.32 MB)

*narration*validation*.jsonl — 1 match(es)
/

In [13]:
import json
from pathlib import Path
from collections import Counter

TEST_FILE = Path(
    "/kaggle/working/SceneOps_recovery_backup/dataset/"
    "sceneops_narration_test_v4.jsonl"
)

assert TEST_FILE.exists(), "Recovery test dataset not found"

with TEST_FILE.open("r", encoding="utf-8") as f:
    test_rows = [json.loads(line) for line in f if line.strip()]

print("Test examples:", len(test_rows))
print("First example keys:", list(test_rows[0].keys()))

print("\nFirst example:")
print(json.dumps(test_rows[0], indent=2, ensure_ascii=False)[:4000])

print("\nField types:")
for key, value in test_rows[0].items():
    print(f"{key}: {type(value).__name__}")

print("\nScenario-related fields:")
for key in test_rows[0]:
    if "scenario" in key.lower():
        print(key, "unique values:", len(set(str(r.get(key)) for r in test_rows)))

assert len(test_rows) > 0
print("\nTEST DATASET LOADED")

Test examples: 1181
First example keys: ['messages']

First example:
{
  "messages": [
    {
      "role": "system",
      "content": "You narrate a synthetic educational mass-casualty training scene. Use only learner-recorded observations supplied in the user message. Do not infer causes, injuries, symptoms, diagnoses, triage categories, or events. Do not reveal hidden simulator state."
    },
    {
      "role": "user",
      "content": "Summarize the learner-recorded observations for patient P003 at simulation second 600: respiratory rate: 30; radial pulse: present; capillary refill: 1.2; mental status: unresponsive; ambulatory status: non_ambulatory; visible hemorrhage: no visible major hemorrhage; follows commands: does not follow commands."
    },
    {
      "role": "assistant",
      "content": "At simulation second 600, the following observations were recorded for patient P003. The recorded respiratory rate was 30 breaths per minute. A radial pulse was present. Capillary refil

In [14]:
import json
import time
import requests
from pathlib import Path

TEST_FILE = Path(
    "/kaggle/working/SceneOps_recovery_backup/dataset/"
    "sceneops_narration_test_v4.jsonl"
)

OUTPUT_DIR = Path("/kaggle/working/SceneOps/evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with TEST_FILE.open("r", encoding="utf-8") as f:
    test_rows = [json.loads(line) for line in f if line.strip()]

results = []
failures = 0

for i, row in enumerate(test_rows[:50], start=1):
    messages = row["messages"]
    user_message = next(m["content"] for m in messages if m["role"] == "user")
    reference = next(m["content"] for m in messages if m["role"] == "assistant")

    try:
        response = requests.post(
            "http://127.0.0.1:8000/generate",
            json={"prompt": user_message},
            timeout=120
        )
        response.raise_for_status()
        prediction = response.json()["generated_text"].strip()

        results.append({
            "example": i,
            "prompt": user_message,
            "reference": reference,
            "prediction": prediction,
            "http_status": response.status_code
        })

    except Exception as e:
        failures += 1
        results.append({
            "example": i,
            "prompt": user_message,
            "reference": reference,
            "prediction": "",
            "error": repr(e)
        })

    if i % 10 == 0:
        print(f"Completed {i}/50 | API failures: {failures}")

output_file = OUTPUT_DIR / "awq_test_pilot_50.jsonl"

with output_file.open("w", encoding="utf-8") as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

successful = [r for r in results if r.get("prediction")]

print("\n========== PILOT SUMMARY ==========")
print("Requested:", 50)
print("Successful:", len(successful))
print("API failures:", failures)
print("Saved:", output_file)

print("\n========== FIRST 3 COMPARISONS ==========")
for item in successful[:3]:
    print("\nREFERENCE:\n", item["reference"])
    print("\nPREDICTION:\n", item["prediction"])
    print("-" * 70)

(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:27:28,176 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 6fad2154-d777-4f4b-b13c-91f9699a92b2 -- POST /generate 200 4676.3ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:27:34,769 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 7e1c472b-c717-4f92-96a2-768d892b4b49 -- POST /generate 200 6587.3ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:27:39,248 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 204e8d9d-a9f8-43c4-8b50-5b2d67190e7f -- POST /generate 200 4472.8ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:27:44,426 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 873b6676-8d0e-4978-aed8-e53d29e5a7fc -- POST /generate 200 5171.5ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:27:50,307 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 db928348-0d02-4174-9422-0fdf2ef4d0aa -- POST /ge

Completed 10/50 | API failures: 0


(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:28:14,492 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 243c3888-21e3-4b10-9f40-8ff649ce867f -- POST /generate 200 4490.5ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:28:19,441 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 4834af18-9cea-49ee-91c5-36896f3a1f55 -- POST /generate 200 4943.3ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:28:23,879 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 9828ad26-4af8-43ed-a623-4c10218acba7 -- POST /generate 200 4431.5ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:28:28,505 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 607c5b58-2fa3-42c2-b5a7-82e29481b412 -- POST /generate 200 4620.5ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:28:33,072 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 c75b03a8-6ff9-4eb3-9d76-7ccca4eaced7 -- POST /ge

Completed 20/50 | API failures: 0


(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:29:02,741 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 1117fc41-afee-46b1-9125-ccfe572c3342 -- POST /generate 200 4711.7ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:29:08,061 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 f8ccf50f-c094-4f00-a903-51f8aa791030 -- POST /generate 200 5314.3ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:29:12,693 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 2391bb21-e463-471d-b2a5-edf2a2672356 -- POST /generate 200 4625.7ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:29:17,959 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 83614fac-f358-4815-83c5-50f58b50a48e -- POST /generate 200 5259.7ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:29:22,708 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 e16a01c2-7bb6-4cf1-a0ef-89ddd445b132 -- POST /ge

Completed 30/50 | API failures: 0


(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:29:51,314 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 65b6cc6a-4899-4ccc-b902-fc14b3e9ed48 -- POST /generate 200 4476.1ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:29:56,459 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 b177baf0-083f-4c07-be31-a3a842f9577c -- POST /generate 200 5138.8ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:30:05,441 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 869fc43f-a4a7-45e5-add6-c29c1c414a01 -- POST /generate 200 8976.4ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:30:09,921 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 32a4d3c7-e8c7-4406-b544-fbd9a9f221f3 -- POST /generate 200 4474.2ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:30:15,154 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 4ed6c554-e931-46c9-b1f5-1443e17b5527 -- POST /ge

Completed 40/50 | API failures: 0


(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:30:45,655 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 25e0e179-a353-4199-9fb2-af6b9d759a97 -- POST /generate 200 4500.1ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:30:50,386 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 5cfa7745-f320-4b5a-9c79-9c24ec18cf74 -- POST /generate 200 4724.6ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:30:57,693 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 17d8fcdd-0434-4bbb-890e-bbf142890293 -- POST /generate 200 7298.6ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:31:02,217 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 6acac63a-b901-47e4-b41e-77b0e81278cb -- POST /generate 200 4518.6ms
(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:31:07,289 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 bd228f02-77d2-4774-8952-39fd76211171 -- POST /ge

Completed 50/50 | API failures: 0

========== PILOT SUMMARY ==========
Requested: 50
Successful: 50
API failures: 0
Saved: /kaggle/working/SceneOps/evaluation/awq_test_pilot_50.jsonl

========== FIRST 3 COMPARISONS ==========

REFERENCE:
 At simulation second 600, the following observations were recorded for patient P003. The recorded respiratory rate was 30 breaths per minute. A radial pulse was present. Capillary refill was recorded as 1.2 seconds. The recorded mental status was unresponsive. The patient was not ambulatory. No visible major hemorrhage was recorded. The patient did not follow commands.

PREDICTION:
 At simulation second 600, the following observations were recorded for patient P003. The recorded respiratory rate was 30 breaths per minute. A radial pulse was present. Capillary refill was recorded as 1.2 seconds. The recorded mental status was unresponsive. The patient was not ambulatory. No visible major hemorrhage was recorded. The patient did not follow commands.
---

(ServeReplica:sceneops-narrator:SceneOpsNarratorV2 pid=5565) INFO 2026-11-19 12:31:36,338 sceneops-narrator_SceneOpsNarratorV2 2fmpe1e1 5748fe93-c5bb-49f0-86e7-9385c9674c39 -- POST /generate 200 5081.3ms


In [15]:
import json
import re
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter

RESULT_FILE = Path(
    "/kaggle/working/SceneOps/evaluation/awq_test_pilot_50.jsonl"
)

with RESULT_FILE.open("r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9.]+", " ", text)
    return " ".join(text.split())

def tokens(text):
    return re.findall(r"\b[\w.]+\b", normalize(text))

exact_matches = 0
similarities = []
precisions = []
recalls = []
f1_scores = []
suspicious = []

suspicious_patterns = [
    r"diagnos",
    r"triage categor",
    r"treatment",
    r"additional information",
    r"medical history",
    r"likely",
    r"probably",
    r"may indicate",
    r"suggests that"
]

for record in records:
    reference = record.get("reference", "")
    prediction = record.get("prediction", "")

    ref_norm = normalize(reference)
    pred_norm = normalize(prediction)

    if ref_norm == pred_norm:
        exact_matches += 1

    similarities.append(
        SequenceMatcher(None, ref_norm, pred_norm).ratio()
    )

    ref_tokens = tokens(reference)
    pred_tokens = tokens(prediction)

    ref_counts = Counter(ref_tokens)
    pred_counts = Counter(pred_tokens)

    overlap = sum((ref_counts & pred_counts).values())

    precision = overlap / max(len(pred_tokens), 1)
    recall = overlap / max(len(ref_tokens), 1)
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    precisions.append(precision)
    recalls.append(recall)
    f1_scores.append(f1)

    flags = [
        pattern for pattern in suspicious_patterns
        if re.search(pattern, prediction, re.IGNORECASE)
    ]

    if flags:
        suspicious.append({
            "example": record["example"],
            "flags": flags,
            "prediction": prediction
        })

n = len(records)

print("=" * 55)
print("SCENEOPS AWQ — 50 EXAMPLE PILOT METRICS")
print("=" * 55)
print(f"Total examples:       {n}")
print(f"API failures:         {sum('error' in r for r in records)}")
print(f"Exact matches:        {exact_matches}/{n} ({exact_matches/n*100:.2f}%)")
print(f"Mean text similarity: {sum(similarities)/n:.4f}")
print(f"Mean token precision: {sum(precisions)/n:.4f}")
print(f"Mean token recall:    {sum(recalls)/n:.4f}")
print(f"Mean token F1:        {sum(f1_scores)/n:.4f}")
print(f"Suspicious outputs:   {len(suspicious)}")

print("\nSUSPICIOUS OUTPUT DETAILS:")
if suspicious:
    for item in suspicious:
        print(f"\nExample {item['example']} | Flags: {item['flags']}")
        print(item["prediction"])
else:
    print("No suspicious phrase patterns detected.")

metrics = {
    "evaluation_type": "50-example held-out AWQ pilot",
    "total_examples": n,
    "api_failures": sum("error" in r for r in records),
    "exact_matches": exact_matches,
    "exact_match_rate": exact_matches / max(n, 1),
    "mean_sequence_similarity": sum(similarities) / max(n, 1),
    "mean_token_precision": sum(precisions) / max(n, 1),
    "mean_token_recall": sum(recalls) / max(n, 1),
    "mean_token_f1": sum(f1_scores) / max(n, 1),
    "suspicious_output_count": len(suspicious)
}

metrics_file = RESULT_FILE.parent / "awq_test_pilot_metrics.json"
metrics_file.write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8"
)

print("\nMetrics saved:", metrics_file)

SCENEOPS AWQ — 50 EXAMPLE PILOT METRICS
Total examples:       50
API failures:         0
Exact matches:        24/50 (48.00%)
Mean text similarity: 0.8150
Mean token precision: 0.9466
Mean token recall:    0.9326
Mean token F1:        0.9383
Suspicious outputs:   0

SUSPICIOUS OUTPUT DETAILS:
No suspicious phrase patterns detected.

Metrics saved: /kaggle/working/SceneOps/evaluation/awq_test_pilot_metrics.json


In [16]:
from pathlib import Path
import json
import shutil
import zipfile
import os

import wandb
from ray import serve
import ray

WORK = Path("/kaggle/working/SceneOps")
EVAL = WORK / "evaluation"
BUNDLE = WORK / "SceneOps_delivery_bundle.zip"

# 1. Stop serving processes after successful API tests
try:
    serve.shutdown()
    print("Ray Serve shut down.")
except Exception as e:
    print("Serve shutdown note:", str(e))

try:
    ray.shutdown()
    print("Ray shut down.")
except Exception as e:
    print("Ray shutdown note:", str(e))

# 2. Load recorded metrics
training_metrics = json.loads(
    (WORK / "final_model" / "training_metrics.json").read_text()
)
pilot_metrics = json.loads(
    (EVAL / "awq_test_pilot_metrics.json").read_text()
)

# 3. W&B offline run (no API key available)
os.environ["WANDB_MODE"] = "offline"

run = wandb.init(
    project="SceneOps-TriageCommand",
    name="qwen25-1.5b-awq-pilot",
    config={
        "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
        "fine_tuning": "QLoRA",
        "quantization": ["AWQ W4A16", "GGUF Q4_K_M"],
        "evaluation_type": "50-example held-out pilot",
        "wandb_sync": "offline"
    }
)

wandb.log({
    "training/train_loss": training_metrics["train_loss"],
    "training/train_runtime_seconds": training_metrics["train_runtime"],
    "training/epochs": training_metrics["epoch"],
    "evaluation/pilot_examples": pilot_metrics["total_examples"],
    "evaluation/exact_match_rate": pilot_metrics["exact_match_rate"],
    "evaluation/mean_sequence_similarity": pilot_metrics["mean_sequence_similarity"],
    "evaluation/token_precision": pilot_metrics["mean_token_precision"],
    "evaluation/token_recall": pilot_metrics["mean_token_recall"],
    "evaluation/token_f1": pilot_metrics["mean_token_f1"],
    "evaluation/api_failures": pilot_metrics["api_failures"],
})
run.finish()

print("W&B offline run logged.")

# 4. Write delivery README
readme = """# SceneOps — TriageCommand

## Project
Medical mass-casualty training narration system.
The simulator remains the source of truth. The language model narrates
only observations supplied to it.

## Model
- Base: Qwen/Qwen2.5-1.5B-Instruct
- Fine-tuning: QLoRA
- Training steps: 669
- Epochs: 3

## Artifacts
- final_model/: LoRA adapter and tokenizer files
- awq_w4a16/: AWQ W4A16 quantized model
- gguf/sceneops-q4_k_m.gguf: GGUF Q4_K_M model
- evaluation/: 50-example held-out pilot outputs and metrics

## Verified
- Fine-tuning completed at global step 669.
- AWQ conversion completed; inference smoke test passed.
- GGUF Q4_K_M conversion completed; inference smoke test passed.
- Ray Serve HTTP health and generation endpoints returned HTTP 200.
- Corrected chat-template API test preserved all six checked observations.
- Held-out AWQ pilot: 50 examples, 0 API failures.

## Pilot metrics
- Exact match: 48%
- Mean sequence similarity: 0.8150
- Token precision: 0.9466
- Token recall: 0.9326
- Token F1: 0.9383

These are lexical pilot metrics, not clinical correctness measurements.
Full 1,181-example held-out evaluation remains pending.
The suspicious-output scan was a basic keyword scan, not a complete safety audit.

## W&B
Metrics logged in offline mode because WANDB_API_KEY was not configured.
The local run must be synced after authenticating with W&B.

## Important limitations
- Do not use this model for real-world medical decisions.
- The simulator, not the LLM, controls patient state and triage logic.
- Training/evaluation loss and token accuracy should be independently audited.
"""

(WORK / "README_delivery.md").write_text(readme, encoding="utf-8")

# 5. Inventory selected artifacts
required = [
    WORK / "SceneOps_final_model.zip",
    WORK / "awq_w4a16",
    WORK / "gguf" / "sceneops-q4_k_m.gguf",
    EVAL / "awq_test_pilot_50.jsonl",
    EVAL / "awq_test_pilot_metrics.json",
]

for path in required:
    assert path.exists(), f"Required artifact missing: {path}"

# 6. Create compact delivery bundle
with zipfile.ZipFile(BUNDLE, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    z.write(WORK / "README_delivery.md", "README_delivery.md")
    z.write(WORK / "SceneOps_final_model.zip", "adapter/SceneOps_final_model.zip")

    for path in (WORK / "awq_w4a16").rglob("*"):
        if path.is_file():
            z.write(path, str(Path("awq_w4a16") / path.relative_to(WORK / "awq_w4a16")))

    z.write(
        WORK / "gguf" / "sceneops-q4_k_m.gguf",
        "gguf/sceneops-q4_k_m.gguf"
    )
    z.write(EVAL / "awq_test_pilot_50.jsonl", "evaluation/awq_test_pilot_50.jsonl")
    z.write(EVAL / "awq_test_pilot_metrics.json", "evaluation/awq_test_pilot_metrics.json")

# 7. Verify bundle integrity
with zipfile.ZipFile(BUNDLE, "r") as z:
    bad_file = z.testzip()
    assert bad_file is None, f"Corrupt file in bundle: {bad_file}"
    print("Bundle entries:", len(z.namelist()))

print("\nDELIVERY BUNDLE VERIFIED")
print("Path:", BUNDLE)
print("Size (GB):", round(BUNDLE.stat().st_size / (1024**3), 3))
print("Free disk (GB):", round(shutil.disk_usage("/kaggle/working").free / (1024**3), 2))
print("W&B offline directory:", Path("/kaggle/working/wandb").exists())

(ServeController pid=5563) INFO 2026-11-19 12:33:47,286 controller 5563 -- Removing 1 replica from Deployment(name='SceneOpsNarratorV2', app='sceneops-narrator').
(ServeController pid=5563) INFO 2026-11-19 12:33:49,310 controller 5563 -- Replica(id='2fmpe1e1', deployment='SceneOpsNarratorV2', app='sceneops-narrator') is stopped.


(raylet) Task ServeController.graceful_shutdown failed. There are infinite retries remaining, so the task will be retried. Error: The actor is dead because it was killed by `ray.kill`.
Ray Serve shut down.
Ray shut down.


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


evaluation/api_failures,▁
evaluation/exact_match_rate,▁
evaluation/mean_sequence_similarity,▁
evaluation/pilot_examples,▁
evaluation/token_f1,▁
evaluation/token_precision,▁
evaluation/token_recall,▁
training/epochs,▁
training/train_loss,▁
training/train_runtime_seconds,▁
evaluation/api_failures,0


W&B offline run logged.
Bundle entries: 12

DELIVERY BUNDLE VERIFIED
Path: /kaggle/working/SceneOps/SceneOps_delivery_bundle.zip
Size (GB): 1.882
Free disk (GB): 8.78
W&B offline directory: True


In [17]:
from pathlib import Path
from IPython.display import display, FileLink

bundle = Path("/kaggle/working/SceneOps/SceneOps_delivery_bundle.zip")

assert bundle.exists(), "Delivery ZIP missing!"
print("File size:", round(bundle.stat().st_size / (1024**3), 3), "GB")
display(FileLink(str(bundle)))

File size: 1.882 GB


/kaggle/working/SceneOps/SceneOps_delivery_bundle.zip

In [18]:
import os
from pathlib import Path
from IPython.display import display, FileLink

os.chdir("/kaggle/working")

file_path = Path("SceneOps/SceneOps_delivery_bundle.zip")

print("Exists:", file_path.exists())
print("Size (GB):", round(file_path.stat().st_size / (1024**3), 3))

display(FileLink("SceneOps/SceneOps_delivery_bundle.zip"))

Exists: True
Size (GB): 1.882


/kaggle/working/SceneOps/SceneOps_delivery_bundle.zip